# Event Attention-Shape Clustering with Sliding SAX

Areproducible statistical pipeline: data audit, preprocessing, symbolic representation, clustering, internal validation, external market validation, and robustness checks designed to verify whether the recovered clusters represent stable attention shapes rather than tuning artifacts.

## Validation roadmap added in this version

1. Test whether fine clusters nest cleanly inside macro clusters (`k=2` vs. `k=9`).
2. Supplement consensus CDF delta with PAC and threshold sensitivity.
3. Extend SAX grid robustness across multiple SAX settings and step fractions.
4. Run preprocessing constant sensitivity for detrending and denoising windows.
5. Define medoids and stable-core terms for cleaner cluster storytelling and prediction tests.
6. Audit whether predictive clusters are driven by many independent events or a few long event episodes.
7. Run same-size placebo cluster draws for an empirical data-mining check.


# Setup and Functions

## 0. Imports and configuration


In [2]:
from __future__ import annotations

import json
import warnings
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------------

DATA_DIR = Path(r"C:\Python\CSUREMM\data_events")
OUTPUT_DIR = Path(r"C:\Python\CSUREMM\output\sax_tests_july_08")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Current folder structure:
# DATA_DIR/
#     query_metadata.csv
#     weather/
#         chunks/
#         diagnostics/
#         stitched/
#             gt_fixed02_weather_2022-01-01_2026-05-31.csv
STITCHED_SUBDIR = "stitched"
STITCHED_GLOB = "gt_fixed02_*.csv"

# Preprocessing controls.
MAX_INTERPOLATE_GAP = 7       # fill missing gaps up to one week
DENOISE_WINDOW = 15           # odd integer; light smoothing only
DENOISE_POLYORDER = 2
DETREND_WINDOW = 91           # 13-week rolling median trend

# SAX tuning grid. PAA word length w is named n_segments in the code.
WINDOW_GRID = [60, 90, 180] # removed 365 days because it was too coarse for the 2022-2026 data
STEP_FRACTIONS = [0.25]       # step = round(window_size * fraction)
PAA_WORD_LENGTH_GRID = [8, 12, 16, 24, 32, 52]
ALPHABET_GRID = [3, 4, 5, 6]

# Clustering grid. This is evaluated after a SAX representation is built.
K_GRID = range(2, 8)
RANDOM_STATE = 42

# --- Anti-degeneracy / balance controls -------------------------------------
# A k=2 split that isolates a handful of outlier terms from everything else
# is statistically "clean" (high silhouette) but not interpretable. These
# controls stop the autoselection stage from rewarding that kind of split.
MIN_CLUSTER_FRACTION = 0.08   # smallest allowed cluster, as a fraction of n_terms
BALANCE_METRIC = "norm_entropy"  # "norm_entropy" or "min_frac"

# --- Multi-resolution (layered) clustering -----------------------------------
LAYER_KS = (3, 6, 10)   # coarse -> meso -> fine cuts of the same dendrogram

# --- Consensus clustering -----------------------------------------------------
CONSENSUS_K_RANGE = range(2, 11)
CONSENSUS_N_RESAMPLES = 200
CONSENSUS_SUBSAMPLE_FRAC = 0.8

# Optional spike features summarize the full preprocessed curve in addition to symbolic shape.
INCLUDE_SPIKE_FEATURES = False

# Efficiency controls for tuning.
# Silhouette is O(n_terms^2); sampling makes grid search much faster on large panels.
TUNING_KMEANS_N_INIT = 10
FINAL_KMEANS_N_INIT = 50
SILHOUETTE_SAMPLE_SIZE = 500
USE_WINDOW_CACHE = True


## 1. Load stitched daily series

The file of interest is the stitched daily series produced from the `chunks/` folder. The raw chunk files remain important for diagnostics, but the clustering input is the stitched output.

This is the practical compromise between the two external references:

- trendEcon emphasizes that long Google Trends series need careful scaling across time windows.
- The current project already performed overlap-rolling stitching, so we avoid re-querying or re-normalizing every chunk.
- The Djorno replication motivates the next stage: statistical preprocessing before modeling/clustering.


In [3]:
def load_all_series(
    data_dir: Path,
    stitched_subdir: str = STITCHED_SUBDIR,
    filename_glob: str = STITCHED_GLOB,
) -> dict[str, pd.Series]:
    """
    Load one stitched daily Google Trends CSV per query folder.

    Expected format:
        DATA_DIR/<query>/stitched/gt_fixed02_<query>_<start>_<end>.csv

    The first non-Time column is treated as the query's value column.
    """
    series_dict: dict[str, pd.Series] = {}
    failed: list[tuple[str, str]] = []

    for folder in sorted(data_dir.iterdir()):
        if not folder.is_dir():
            continue

        stitched_dir = folder / stitched_subdir
        candidates = sorted(stitched_dir.glob(filename_glob))

        if not candidates:
            failed.append((folder.name, f"missing stitched csv in {stitched_dir}"))
            continue

        fpath = candidates[-1]

        try:
            df = pd.read_csv(fpath, parse_dates=["Time"])
            value_cols = [c for c in df.columns if c != "Time"]
            if not value_cols:
                raise ValueError("no value column found")

            value_col = value_cols[0]
            ts = (
                df[["Time", value_col]]
                .dropna(subset=["Time"])
                .drop_duplicates("Time")
                .sort_values("Time")
                .set_index("Time")[value_col]
                .astype(float)
            )
            ts.name = folder.name
            series_dict[folder.name] = ts

        except Exception as e:
            failed.append((folder.name, str(e)))

    print(f"Loaded {len(series_dict)} stitched series.")
    if failed:
        print(f"Failed to load {len(failed)} folders:")
        for name, err in failed[:25]:
            print(f"  - {name}: {err}")
        if len(failed) > 25:
            print(f"  ... {len(failed) - 25} more")

    return series_dict


def build_panel(series_dict: dict[str, pd.Series]) -> pd.DataFrame:
    """Align all query series on a common daily date index."""
    if not series_dict:
        raise ValueError("series_dict is empty")

    panel = pd.DataFrame(series_dict).sort_index()
    full_index = pd.date_range(panel.index.min(), panel.index.max(), freq="D")
    return panel.reindex(full_index)


def basic_quality_report(panel: pd.DataFrame) -> pd.DataFrame:
    """Basic post-stitch diagnostics per query."""
    rows = []
    for col in panel.columns:
        x = panel[col]
        rows.append({
            "query": col,
            "n_days_total": len(x),
            "n_nonmissing": int(x.notna().sum()),
            "missing_share": float(x.isna().mean()),
            "zero_share": float((x == 0).mean(skipna=True)),
            "min": float(x.min(skipna=True)),
            "max": float(x.max(skipna=True)),
            "mean": float(x.mean(skipna=True)),
            "std": float(x.std(skipna=True)),
            "range": float(x.max(skipna=True) - x.min(skipna=True)),
        })
    return pd.DataFrame(rows).sort_values("query")


## 2. Post-stitch preprocessing

The preprocessing layer adapts the Djorno et al. logic to this project:

1. fill short missing gaps;
2. apply light denoising, so SAX does not overreact to one-day noise;
3. remove slow-moving trend, so clustering focuses on attention shape;
4. robustly normalize each query once over the full daily series.

This normalization is **not per chunk** and **not per SAX window**. Window-level z-normalization still occurs inside SAX so each local word describes shape rather than absolute level.


In [4]:
def interpolate_small_gaps(s: pd.Series, max_gap: int = MAX_INTERPOLATE_GAP) -> pd.Series:
    """Interpolate short missing gaps without filling long absent regions."""
    return s.astype(float).interpolate(
        method="time",
        limit=max_gap,
        limit_direction="both",
    )


def denoise_series(
    s: pd.Series,
    window: int = DENOISE_WINDOW,
    polyorder: int = DENOISE_POLYORDER,
) -> pd.Series:
    """Light Savitzky-Golay denoising. Falls back to rolling median for short series."""
    x = s.astype(float)
    y = x.copy()
    valid = x.dropna()
    if len(valid) < 5:
        return x
    win = min(window, len(valid) if len(valid) % 2 == 1 else len(valid) - 1)
    if win <= polyorder + 2:
        return x.rolling(7, center=True, min_periods=1).median()
    if win % 2 == 0:
        win -= 1
    filled = interpolate_small_gaps(x).ffill().bfill()
    smoothed = savgol_filter(filled.values, window_length=win, polyorder=polyorder)
    y.loc[:] = smoothed
    y[x.isna()] = np.nan
    return y.rename(s.name)


def detrend_series(s: pd.Series, window: int = DETREND_WINDOW) -> pd.Series:
    """Remove slow trend using a centered rolling median."""
    x = s.astype(float)
    trend = x.rolling(window=window, center=True, min_periods=max(7, window // 4)).median()
    trend = trend.bfill().ffill()
    return (x - trend).rename(s.name)


def robust_zscore_series(
    s: pd.Series,
    mad_floor_frac: float = 0.05,
    zero_share_threshold: float = 0.30,
    mad_ratio_threshold: float = 0.02,
    clip_mad_multiple: float = 4.0,
) -> pd.Series:
    """
    Robust per-query normalization using median and MAD.

    Only applies a MAD floor + winsorization for series that actually look
    like the degenerate "near-empty baseline + rare spike" case (high
    zero-share and/or a MAD that is tiny relative to the series' own spread).
    Well-behaved series pass through with plain MAD z-scoring, untouched.
    """
    x = s.astype(float)
    med = x.median(skipna=True)
    mad = (x - med).abs().median(skipna=True)

    if pd.isna(mad) or mad == 0:
        std = x.std(skipna=True)
        if pd.isna(std) or std == 0:
            return pd.Series(np.zeros(len(x)), index=x.index, name=x.name)
        return ((x - x.mean(skipna=True)) / std).rename(x.name)

    # tail-aware spread, used only to detect/floor the degenerate case
    p05, p95 = x.quantile(0.05), x.quantile(0.95)
    wide_spread = (p95 - p05) / 3.29

    zero_share = float((x == 0).mean())
    mad_ratio = (mad / wide_spread) if wide_spread > 0 else 1.0

    is_degenerate = mad_ratio < 0.05

    if not is_degenerate:
        # normal path: untouched MAD z-score, no floor, no clip
        return (0.6745 * (x - med) / mad).rename(x.name)

    # degenerate path: floor the scale, then winsorize relative to the
    # (floored) scale itself rather than an absolute cutoff, so the clip
    # adapts to each series instead of chopping every series at the same point
    mad_floored = max(mad, mad_floor_frac * wide_spread) if wide_spread > 0 else mad
    z = (0.6745 * (x - med) / mad_floored).rename(x.name)
    return z.clip(lower=-clip_mad_multiple, upper=clip_mad_multiple)

def soft_cap_series(s: pd.Series, cap: float = 10.0) -> pd.Series:
    return (cap * np.tanh(s / cap)).rename(s.name)

def soft_cap_panel(df: pd.DataFrame, cap: float = 10.0) -> pd.DataFrame:
    return df.apply(soft_cap_series, axis=0, cap=cap)

def preprocess_panel(panel: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    """
    Apply the current preprocessing pipeline to all columns.

    Returns
    -------
    panel_preprocessed:
        daily panel after interpolation, denoising, detrending, and robust z-score.
    stages:
        intermediate panels useful for diagnostics and plotting.
    """
    filled = panel.apply(interpolate_small_gaps, axis=0)
    denoised = filled.apply(denoise_series, axis=0)
    detrended = denoised.apply(detrend_series, axis=0)
    normalized = detrended.apply(robust_zscore_series, axis=0)
    normalized_soft = soft_cap_panel(
         normalized,
         cap=10.0
    )
    
    stages = {
        "filled": filled,
        "denoised": denoised,
        "detrended": detrended,
        "normalized": normalized,
        "soft": normalized_soft
    }
    return normalized, stages

## 3. Sliding SAX representation

The three retained tuning dimensions are implemented here:

- `window_size`: the local time horizon summarized by one SAX word;
- `step_size`: how often the window moves forward;
- `PAA word length w`: implemented as `n_segments`;
- `alphabet_size a`: number of vertical states.

For example, `window_size = 180`, `step = 30`, `n_segments = 24`, and `alphabet_size = 4` means each 180-day local episode is compressed into a 24-symbol word using four vertical levels.


In [ ]:
def gaussian_breakpoints(alphabet_size: int) -> np.ndarray:
    """Gaussian SAX breakpoints for equiprobable symbolic bins."""
    return stats.norm.ppf(np.linspace(0, 1, alphabet_size + 1)[1:-1])


def valid_sax_setting(window_size: int, step: int, n_segments: int, alphabet_size: int) -> bool:
    """Keep settings interpretable and numerically valid."""
    if window_size < 2 or step < 1 or n_segments < 2 or alphabet_size < 2:
        return False
        
    if n_segments >= window_size:
        return False
    if window_size / n_segments < 3:
        return False
    return True


def panel_to_array_dict(panel: pd.DataFrame) -> dict[str, np.ndarray]:
    """Convert the normalized panel to compact NumPy arrays once."""
    out: dict[str, np.ndarray] = {}
    for col in panel.columns:
        x = panel[col].dropna().to_numpy(dtype=np.float32)
        if len(x) > 0:
            out[col] = x
    return out


def znormalize_windows_matrix(windows: np.ndarray) -> np.ndarray:
    """Vectorized z-normalization for all sliding windows of one series."""
    mu = windows.mean(axis=1, keepdims=True)
    sigma = windows.std(axis=1, keepdims=True)
    sigma[sigma == 0] = 1.0
    return (windows - mu) / sigma


def paa_matrix(z_windows: np.ndarray, n_segments: int) -> np.ndarray:
    """Vectorized PAA using cumulative sums for all windows at once."""
    n_windows, window_size = z_windows.shape
    bounds = np.linspace(0, window_size, n_segments + 1, dtype=np.int64)
    csum = np.concatenate(
        [np.zeros((n_windows, 1), dtype=z_windows.dtype), np.cumsum(z_windows, axis=1)],
        axis=1,
    )
    seg_sum = csum[:, bounds[1:]] - csum[:, bounds[:-1]]
    seg_len = (bounds[1:] - bounds[:-1]).astype(z_windows.dtype)
    return seg_sum / seg_len


def make_window_cache(
    array_dict: dict[str, np.ndarray],
    settings: list[dict[str, int]],
) -> dict[tuple[int, int], dict[str, np.ndarray]]:
    """Cache sliding windows for every distinct `(window_size, step)` pair."""
    cache: dict[tuple[int, int], dict[str, np.ndarray]] = {}
    pairs = sorted({(s["window_size"], s["step"]) for s in settings})

    for window_size, step in pairs:
        term_windows: dict[str, np.ndarray] = {}
        for term, x in array_dict.items():
            if len(x) < window_size:
                continue
            windows = sliding_window_view(x, window_size)[::step].astype(np.float32, copy=True)
            if len(windows) > 0:
                term_windows[term] = windows
        cache[(window_size, step)] = term_windows

    return cache


def build_sax_feature_matrix_fast(
    array_dict: dict[str, np.ndarray],
    window_size: int,
    n_segments: int,
    alphabet_size: int,
    step: int,
    spike_features: bool = INCLUDE_SPIKE_FEATURES,
    window_cache: dict[tuple[int, int], dict[str, np.ndarray]] | None = None,
) -> pd.DataFrame:
    """
    Fast query-level SAX profile construction.

    Feature design is unchanged:
    - mean symbolic level for each PAA segment across sliding windows;
    - distribution of symbols across the whole query;
    - optional spike features from the normalized preprocessed series.
    """
    if not valid_sax_setting(window_size, step, n_segments, alphabet_size):
        raise ValueError(
            f"Invalid SAX setting: window={window_size}, step={step}, "
            f"n_segments={n_segments}, alphabet={alphabet_size}"
        )

    breakpoints = gaussian_breakpoints(alphabet_size).astype(np.float32)
    cached_windows = None if window_cache is None else window_cache.get((window_size, step))
    feature_rows = {}

    iterator = cached_windows.items() if cached_windows is not None else array_dict.items()

    for term, obj in iterator:
        if cached_windows is not None:
            windows = obj
            x = array_dict[term]
        else:
            x = obj
            if len(x) < window_size:
                continue
            windows = sliding_window_view(x, window_size)[::step].astype(np.float32, copy=True)
            if len(windows) == 0:
                continue

        z = znormalize_windows_matrix(windows)
        seg_means = paa_matrix(z, n_segments)
        symbols = np.searchsorted(breakpoints, seg_means).astype(np.int16)

        row = {f"sax_seg_mean_{i + 1:02d}": float(v) for i, v in enumerate(symbols.mean(axis=0))}

        counts = np.bincount(symbols.ravel(), minlength=alphabet_size).astype(float)
        shares = counts / counts.sum()
        for a, share in enumerate(shares):
            row[f"symbol_share_{a}"] = float(share)

        if spike_features:
            row.update({
                "p95_z": float(np.percentile(x, 95)),
                "p99_z": float(np.percentile(x, 99)),
                "spike_share_z2": float(np.mean(x > 2)),
                "spike_share_z3": float(np.mean(x > 3)),
            })

        feature_rows[term] = row

    return pd.DataFrame.from_dict(feature_rows, orient="index").dropna(axis=0, how="any")


# Backward-compatible alias.
def build_sax_feature_matrix(*args, **kwargs) -> pd.DataFrame:
    return build_sax_feature_matrix_fast(*args, **kwargs)


## 4. Tune the SAX representation

This stage tunes the SAX representation, not the final interpretation. It evaluates each setting using the best available KMeans score over a small `k` grid.

Recommended selection rule:

1. discard settings with too few usable terms;
2. prefer higher silhouette;
3. prefer lower Davies-Bouldin;
4. among close candidates, choose the simpler/more interpretable representation.

This avoids picking a very high-resolution SAX encoding that only wins because it compresses less.


In [6]:
def make_sax_grid(
    window_grid=WINDOW_GRID,
    step_fractions=STEP_FRACTIONS,
    paa_grid=PAA_WORD_LENGTH_GRID,
    alphabet_grid=ALPHABET_GRID,
) -> list[dict[str, int]]:
    """Construct valid SAX settings from the requested parameter families."""
    settings = []
    for window_size in window_grid:
        for frac in step_fractions:
            step = max(1, int(round(window_size * frac)))
            for n_segments in paa_grid:
                for alphabet_size in alphabet_grid:
                    if valid_sax_setting(window_size, step, n_segments, alphabet_size):
                        settings.append({
                            "window_size": int(window_size),
                            "step": int(step),
                            "n_segments": int(n_segments),
                            "alphabet_size": int(alphabet_size),
                        })
    return settings


def cluster_balance_metrics(labels: np.ndarray) -> dict[str, float]:
    """
    Quantify how balanced a clustering is, independent of geometric separation.

    A k=2 solution that isolates 6% of terms as "outliers" and dumps the rest
    into one blob can have excellent silhouette/Davies-Bouldin scores while
    being useless for interpretation. These metrics let the selection rule
    penalize that pattern instead of rewarding it.
    """
    counts = np.bincount(labels)
    counts = counts[counts > 0]
    n = counts.sum()
    k = len(counts)
    min_frac = float(counts.min() / n)
    probs = counts / n
    entropy = float(-(probs * np.log(probs)).sum())
    max_entropy = np.log(k) if k > 1 else 1.0
    norm_entropy = float(entropy / max_entropy) if max_entropy > 0 else 0.0
    return {"min_cluster_frac": min_frac, "norm_entropy": norm_entropy}


def evaluate_clustering_for_features(
    features: pd.DataFrame,
    k_grid=K_GRID,
    random_state: int = RANDOM_STATE,
    n_init: int = TUNING_KMEANS_N_INIT,
    silhouette_sample_size: int | None = SILHOUETTE_SAMPLE_SIZE,
) -> pd.DataFrame:
    """
    Evaluate MiniBatchKMeans across k for one feature matrix.

    Silhouette is expensive because it uses pairwise distances. During tuning,
    this function samples rows when there are many terms.

    In addition to the geometric scores, this now records balance metrics
    (`min_cluster_frac`, `norm_entropy`) for every k so degenerate splits
    (one dominant cluster + a handful of outliers) can be filtered out
    downstream instead of silently winning on silhouette alone.
    """
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    rows = []

    sample_size = None
    if silhouette_sample_size is not None and len(features) > silhouette_sample_size:
        sample_size = int(silhouette_sample_size)

    valid_ks = [k for k in k_grid if 2 <= k < len(features)]
    for k in valid_ks:
        labels = MiniBatchKMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=n_init,
            batch_size=min(512, len(features)),
            max_iter=100,
        ).fit_predict(X)

        balance = cluster_balance_metrics(labels)

        rows.append({
            "k": int(k),
            "silhouette": float(silhouette_score(
                X,
                labels,
                sample_size=sample_size,
                random_state=random_state,
            )),
            "davies_bouldin": float(davies_bouldin_score(X, labels)),
            "calinski_harabasz": float(calinski_harabasz_score(X, labels)),
            **balance,
        })

    return pd.DataFrame(rows)


def tune_sax_representation(
    array_dict: dict[str, np.ndarray],
    settings: list[dict[str, int]],
    k_grid=K_GRID,
    spike_features: bool = INCLUDE_SPIKE_FEATURES,
    min_terms: int = 10,
    use_window_cache: bool = USE_WINDOW_CACHE,
    min_cluster_fraction: float = MIN_CLUSTER_FRACTION,
    balance_metric: str = BALANCE_METRIC,
) -> pd.DataFrame:
    """
    Grid-search window, step, PAA word length, and alphabet size efficiently.

    Selection fix: previously the "best" k for a setting was whichever k
    (2..7) maximized silhouette, with no check on whether that k actually
    produced usable clusters. A window=365 / n_segments=8 / alphabet=3
    setting can win at k=2 purely because it isolates a few extreme series
    from one giant undifferentiated blob (see 06_kmeans_labels.csv: 363 vs 24
    terms). That is a trivial, low-information split, not a real shape
    distinction -- and it also happens to be the *coarsest* representation on
    the grid, which is why "autoselect the best setting" kept returning the
    least detailed encoding.

    Fix, in two parts:
      1. For each setting, only consider k values whose smallest cluster
         holds at least `min_cluster_fraction` of all terms (or, if using
         entropy, whose normalized entropy clears a floor). Degenerate
         "outlier vs. everything" splits are dropped from contention before
         ranking even starts.
      2. Across settings, drop the old "fewer features wins ties" rule
         (which explicitly biased toward coarser SAX words) and instead use
         balance as the tie-breaker, so a mid-resolution setting that
         produces a genuinely structured partition is preferred over a
         coarse one that only *looks* clean.
    """
    rows = []
    window_cache = make_window_cache(array_dict, settings) if use_window_cache else None

    for i, setting in enumerate(settings, start=1):
        try:
            features = build_sax_feature_matrix_fast(
                array_dict=array_dict,
                window_size=setting["window_size"],
                n_segments=setting["n_segments"],
                alphabet_size=setting["alphabet_size"],
                step=setting["step"],
                spike_features=spike_features,
                window_cache=window_cache,
            )
        except Exception as e:
            rows.append({**setting, "status": f"failed: {e}"})
            continue

        if len(features) < min_terms:
            rows.append({
                **setting,
                "status": "too_few_terms",
                "n_terms": len(features),
            })
            continue

        diag = evaluate_clustering_for_features(features, k_grid=k_grid)
        if diag.empty:
            rows.append({
                **setting,
                "status": "no_valid_k",
                "n_terms": len(features),
            })
            continue

        # --- balance-filtered selection --------------------------------
        if balance_metric == "min_frac":
            balanced = diag[diag["min_cluster_frac"] >= min_cluster_fraction]
        else:
            # a norm_entropy floor of ~0.65 already excludes "one blob + a
            # sliver" splits while still allowing moderately uneven clusters
            balanced = diag[diag["norm_entropy"] >= 0.65]

        candidate_pool = balanced if not balanced.empty else diag
        degenerate_only = balanced.empty

        best = (
            candidate_pool.sort_values(
                ["silhouette", "davies_bouldin"], ascending=[False, True]
            ).iloc[0]
        )

        rows.append({
            **setting,
            "status": "ok" if not degenerate_only else "ok_degenerate_only",
            "n_terms": len(features),
            "n_features": features.shape[1],
            "best_k": int(best["k"]),
            "silhouette": float(best["silhouette"]),
            "davies_bouldin": float(best["davies_bouldin"]),
            "calinski_harabasz": float(best["calinski_harabasz"]),
            "min_cluster_frac": float(best["min_cluster_frac"]),
            "norm_entropy": float(best["norm_entropy"]),
        })

        if i % 10 == 0:
            print(f"  evaluated {i}/{len(settings)} SAX settings")

    results = pd.DataFrame(rows)
    ok = results[results["status"] == "ok"].copy()
    degenerate = results[results["status"] == "ok_degenerate_only"].copy()
    not_ok = results[~results["status"].isin(["ok", "ok_degenerate_only"])].copy()

    # Rank by silhouette / Davies-Bouldin as before, but tie-break on balance
    # (norm_entropy, descending) instead of n_features (ascending). This is
    # the change that stops the routine from defaulting to the coarsest,
    # least-detailed SAX word on the grid.
    if not ok.empty:
        ok = ok.sort_values(
            ["silhouette", "davies_bouldin", "norm_entropy"],
            ascending=[False, True, False],
        )
    if not degenerate.empty:
        degenerate = degenerate.sort_values(
            ["silhouette", "davies_bouldin"], ascending=[False, True]
        )

    # Settings that could only produce degenerate splits at every k are kept
    # for transparency but ranked after genuinely balanced settings, never
    # silently preferred just because their silhouette number is bigger.
    return pd.concat([ok, degenerate, not_ok], ignore_index=True)


## 5. Final clustering and candidate comparison

After selecting a SAX setting, this section builds the final feature matrix and compares a small number of cluster counts. For the clinic brief, prioritize a solution that is both statistically reasonable and easy to describe.


In [7]:
def run_kmeans_final(
    features: pd.DataFrame,
    k: int,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.Series, dict]:
    """Run final KMeans on scaled SAX features."""
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    km = KMeans(n_clusters=k, random_state=random_state, n_init=FINAL_KMEANS_N_INIT)
    labels_array = km.fit_predict(X)
    labels = pd.Series(labels_array, index=features.index, name="cluster")

    metrics = {
        "k": int(k),
        "silhouette": float(silhouette_score(X, labels_array)),
        "davies_bouldin": float(davies_bouldin_score(X, labels_array)),
        "calinski_harabasz": float(calinski_harabasz_score(X, labels_array)),
        "inertia": float(km.inertia_),
    }
    return labels, metrics


def run_ward_final(features: pd.DataFrame, k: int) -> tuple[pd.Series, np.ndarray, dict]:
    """Ward hierarchical clustering for comparison."""
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    Z = linkage(X, method="ward")
    labels_array = fcluster(Z, t=k, criterion="maxclust") - 1
    labels = pd.Series(labels_array, index=features.index, name="cluster")
    metrics = {
        "k": int(k),
        "silhouette": float(silhouette_score(X, labels_array)),
        "davies_bouldin": float(davies_bouldin_score(X, labels_array)),
        "calinski_harabasz": float(calinski_harabasz_score(X, labels_array)),
    }
    return labels, Z, metrics


def cluster_member_table(labels: pd.Series) -> pd.DataFrame:
    return (
        labels.rename_axis("term")
        .reset_index()
        .sort_values(["cluster", "term"])
        .reset_index(drop=True)
    )


def cluster_size_table(labels: pd.Series) -> pd.DataFrame:
    return (
        labels.value_counts()
        .sort_index()
        .rename_axis("cluster")
        .reset_index(name="n_terms")
    )


## 5b. Hierarchical clustering add-on

This optional block runs hierarchical clustering on the same tuned sliding-SAX feature matrix used by KMeans. It is useful as a robustness check because it does not rely on random centroid initialization and produces a dendrogram that can be shown in the clinic brief.

Recommended use:

- keep KMeans as the main scalable baseline;
- use Ward hierarchical clustering as the interpretable comparison;
- compare cluster sizes and validation metrics before deciding which result to present.

In [ ]:
# def run_hierarchical_grid(
#     features: pd.DataFrame,
#     k_grid=K_GRID,
#     methods=("ward", "average", "complete"),
# ) -> pd.DataFrame:
#     """
#     Evaluate hierarchical clustering across linkage methods and k values.

#     Notes:
#     - Ward uses Euclidean geometry and usually works best with standardized continuous features.
#     - Average/complete are included as robustness checks.
#     - The returned table can be compared with KMeans tuning/final metrics.
#     """
#     X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
#     rows = []

#     for method in methods:
#         try:
#             Z = linkage(X, method=method)
#         except Exception as e:
#             rows.append({"method": method, "status": f"failed: {e}"})
#             continue

#         for k in k_grid:
#             if not (2 <= k < len(features)):
#                 continue

#             labels_array = fcluster(Z, t=int(k), criterion="maxclust") - 1

#             # Skip degenerate cuts that do not produce at least two clusters.
#             if len(np.unique(labels_array)) < 2:
#                 continue

#             rows.append({
#                 "method": method,
#                 "k": int(k),
#                 "n_clusters_observed": int(len(np.unique(labels_array))),
#                 "silhouette": float(silhouette_score(X, labels_array)),
#                 "davies_bouldin": float(davies_bouldin_score(X, labels_array)),
#                 "calinski_harabasz": float(calinski_harabasz_score(X, labels_array)),
#                 "status": "ok",
#             })

#     results = pd.DataFrame(rows)
#     ok = results[results["status"] == "ok"].copy()
#     not_ok = results[results["status"] != "ok"].copy()

#     if not ok.empty:
#         ok = ok.sort_values(
#             ["silhouette", "davies_bouldin", "calinski_harabasz"],
#             ascending=[False, True, False],
#         )

#     return pd.concat([ok, not_ok], ignore_index=True)


# def run_hierarchical_final(
#     features: pd.DataFrame,
#     k: int,
#     method: str = "ward",
# ) -> tuple[pd.Series, np.ndarray, dict]:
#     """Run final hierarchical clustering with a chosen linkage method and k."""
#     X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
#     Z = linkage(X, method=method)
#     labels_array = fcluster(Z, t=int(k), criterion="maxclust") - 1
#     labels = pd.Series(labels_array, index=features.index, name="cluster")

#     metrics = {
#         "method": method,
#         "k": int(k),
#         "n_clusters_observed": int(len(np.unique(labels_array))),
#         "silhouette": float(silhouette_score(X, labels_array)),
#         "davies_bouldin": float(davies_bouldin_score(X, labels_array)),
#         "calinski_harabasz": float(calinski_harabasz_score(X, labels_array)),
#     }
#     return labels, Z, metrics


## 5c. Layered (multi-resolution) clustering

A single flat `k` — whatever "autoselect" happens to land on — is exactly
what produces the "one blob + a sliver" result you were seeing. It also
throws away information: attention curves usually have real structure at
more than one resolution (e.g. "seasonal vs. non-seasonal" as a coarse
split, with "which season" or "spike-driven vs. slow-burn" as a finer split
nested inside it).

This section builds **one** Ward dendrogram on the selected SAX features and
cuts it at several heights (`LAYER_KS`, e.g. 3 / 6 / 10 clusters). Because
all cuts come from the same tree, the finer layer is guaranteed to be a
refinement of the coarser layer — every fine cluster sits inside exactly one
coarse cluster. That nesting is what makes the result "interpretable in
layers": you can present the 3-way macro grouping in the headline slide and
drill into the 10-way grouping for the appendix, without the two
contradicting each other.


In [8]:
def build_layered_clusters(
    features: pd.DataFrame,
    layer_ks: tuple[int, ...] = LAYER_KS,
    method: str = "ward",
) -> tuple[pd.DataFrame, np.ndarray]:
    """
    Cut a single hierarchical tree at multiple heights to produce nested,
    multi-resolution cluster labels (coarse -> meso -> fine).

    Returns a DataFrame indexed by term with one column per layer
    (e.g. "cluster_k3", "cluster_k6", "cluster_k10") plus the linkage matrix
    Z, so a dendrogram annotated with layer boundaries can be plotted.
    """
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    Z = linkage(X, method=method)

    layer_cols = {}
    for k in sorted(layer_ks):
        if not (2 <= k < len(features)):
            continue
        labels_array = fcluster(Z, t=int(k), criterion="maxclust") - 1
        layer_cols[f"cluster_k{k}"] = labels_array

    layered = pd.DataFrame(layer_cols, index=features.index)
    layered.index.name = "term"
    return layered, Z


def describe_layer_nesting(layered: pd.DataFrame) -> pd.DataFrame:
    """
    Sanity-check table: for each coarse-layer cluster, how many distinct
    fine-layer clusters sit inside it. If nesting is working, every
    fine-layer id maps to exactly one coarse-layer id.
    """
    cols = sorted(layered.columns, key=lambda c: int(c.split("k")[1]))
    coarsest, finest = cols[0], cols[-1]
    rows = []
    for c in sorted(layered[coarsest].unique()):
        members = layered[layered[coarsest] == c]
        rows.append({
            coarsest: c,
            "n_terms": len(members),
            "n_fine_subclusters": members[finest].nunique(),
        })
    return pd.DataFrame(rows)


## 5d. Consensus clustering (post parameter-selection)

The earlier revision of this notebook removed consensus clustering from the
core workflow. Bringing it back here, but placed *after* SAX parameter
selection rather than mixed into it, because it is solving a different
problem:

- SAX parameter tuning (section 4) asks *which representation* of the
  curves to use.
- Consensus clustering (here) asks, **for a fixed, already-chosen
  representation**, *whether a given cluster structure is stable* — i.e.
  would repeated clustering runs on resampled subsets of the terms agree on
  who belongs with whom, or is the partition an artifact of one particular
  KMeans initialization / subsample?

Method (standard evidence-accumulation / consensus clustering, Monti et al.
2003):

1. For each candidate `k`, repeatedly subsample a fraction of terms,
   cluster the subsample with KMeans, and accumulate a **co-association
   matrix** `C[i, j]` = fraction of resamples in which terms *i* and *j*
   landed in the same cluster (counting only resamples where both were
   drawn).
2. Summarize each `C` with the area under its consensus CDF — near 1.0 means
   almost every pair is either "always together" or "never together"
   (stable); near 0.5 means most pairs are ambiguous (unstable).
3. Pick `k` where that area is high and stops improving much as `k`
   increases further (the classic "delta-K" rule).
4. Build the final labels by clustering on `1 - C` (average linkage)
   instead of on the raw KMeans output, so the reported clusters are the
   ones that were actually stable across resamples, not just one lucky
   draw.
5. Report a per-term **stability score** (average co-association with its
   own cluster minus its next-best cluster) so ambiguous, "in-between"
   terms can be flagged rather than force-assigned.


In [9]:
from scipy.spatial.distance import squareform


def consensus_matrix(
    X: np.ndarray,
    k: int,
    n_resamples: int = CONSENSUS_N_RESAMPLES,
    subsample_frac: float = CONSENSUS_SUBSAMPLE_FRAC,
    random_state: int = RANDOM_STATE,
) -> np.ndarray:
    """Build the co-association (consensus) matrix for one k via subsampling + KMeans."""
    n = X.shape[0]
    rng = np.random.default_rng(random_state)
    M = np.zeros((n, n), dtype=np.float64)   # co-clustered counts
    I = np.zeros((n, n), dtype=np.float64)   # co-sampled counts
    sub_n = max(k + 1, int(round(subsample_frac * n)))

    for _ in range(n_resamples):
        idx = rng.choice(n, size=sub_n, replace=False)
        seed = int(rng.integers(0, 1_000_000))
        labels_sub = KMeans(
            n_clusters=k, n_init=10, random_state=seed
        ).fit_predict(X[idx])

        same = (labels_sub[:, None] == labels_sub[None, :]).astype(np.float64)
        M[np.ix_(idx, idx)] += same
        I[np.ix_(idx, idx)] += 1.0

    with np.errstate(invalid="ignore", divide="ignore"):
        C = np.divide(M, I, out=np.zeros_like(M), where=I > 0)
    np.fill_diagonal(C, 1.0)
    return C


def consensus_cdf_area(C: np.ndarray) -> float:
    """Area under the empirical CDF of off-diagonal consensus values (stability score)."""
    vals = C[np.triu_indices_from(C, k=1)]
    if vals.size == 0:
        return float("nan")
    vals_sorted = np.sort(vals)
    grid = np.linspace(0.0, 1.0, 101)
    cdf = np.searchsorted(vals_sorted, grid, side="right") / len(vals_sorted)
    return float(np.trapezoid(cdf, grid))


def scan_consensus_k(
    features: pd.DataFrame,
    k_range=CONSENSUS_K_RANGE,
    n_resamples: int = CONSENSUS_N_RESAMPLES,
    subsample_frac: float = CONSENSUS_SUBSAMPLE_FRAC,
    random_state: int = RANDOM_STATE,
) -> tuple[pd.DataFrame, dict[int, np.ndarray]]:
    """
    Run consensus clustering across a range of k and summarize stability
    (CDF area, and its first difference) so k can be chosen for genuine
    structural stability rather than a single KMeans run's silhouette.
    """
    X = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
    matrices: dict[int, np.ndarray] = {}
    rows = []
    prev_area = None

    valid_ks = [k for k in k_range if 2 <= k < len(features)]
    for k in valid_ks:
        C = consensus_matrix(
            X, k, n_resamples=n_resamples,
            subsample_frac=subsample_frac, random_state=random_state,
        )
        matrices[k] = C
        area = consensus_cdf_area(C)
        delta = area if prev_area is None else (area - prev_area) / prev_area
        rows.append({"k": k, "consensus_cdf_area": area, "delta_area": delta})
        prev_area = area

    return pd.DataFrame(rows), matrices


def consensus_labels(C: np.ndarray, k: int, index) -> pd.Series:
    """Final cluster assignment from a consensus matrix (average linkage on 1 - C)."""
    dist = 1.0 - C
    np.fill_diagonal(dist, 0.0)
    dist = np.clip(dist, 0, None)
    condensed = squareform(dist, checks=False)
    Z = linkage(condensed, method="average")
    labels_array = fcluster(Z, t=int(k), criterion="maxclust") - 1
    return pd.Series(labels_array, index=index, name="cluster")


def consensus_term_stability(C: np.ndarray, labels: pd.Series) -> pd.Series:
    """
    Per-term stability score: mean co-association with its own cluster
    minus the mean co-association with the best-matching *other* cluster.
    Values near 1 = confidently, consistently clustered. Values near 0 or
    negative = ambiguous / borderline term worth flagging in the write-up.
    """
    labels_arr = labels.to_numpy()
    scores = np.zeros(len(labels_arr))
    unique_clusters = np.unique(labels_arr)

    for i in range(len(labels_arr)):
        own = labels_arr[i]
        own_mask = (labels_arr == own)
        own_mask[i] = False
        own_mean = C[i, own_mask].mean() if own_mask.any() else np.nan

        other_means = []
        for c in unique_clusters:
            if c == own:
                continue
            mask = (labels_arr == c)
            if mask.any():
                other_means.append(C[i, mask].mean())
        best_other = max(other_means) if other_means else 0.0
        scores[i] = own_mean - best_other

    return pd.Series(scores, index=labels.index, name="stability_score")


In [10]:
def plot_cluster_average_curves(
    panel_norm: pd.DataFrame,
    labels: pd.Series,
    outpath: Path,
    title: str = "Cluster-average normalized attention curves",
):
    """Plot one average curve per cluster."""
    plt.figure(figsize=(12, 6))
    for c in sorted(labels.unique()):
        members = labels[labels == c].index.intersection(panel_norm.columns)
        if len(members) == 0:
            continue
        avg = panel_norm[members].mean(axis=1, skipna=True)
        plt.plot(avg.index, avg.values, linewidth=1.8, label=f"Cluster {c} (n={len(members)})")

    plt.axhline(0, linewidth=0.8)
    plt.title(title)
    plt.ylabel("Robust z-score after denoising and detrending")
    plt.xlabel("Date")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.close()


def plot_dendrogram_for_features(features: pd.DataFrame, Z: np.ndarray, outpath: Path):
    plt.figure(figsize=(14, 6))
    dendrogram(Z, labels=features.index.tolist(), leaf_rotation=90, leaf_font_size=6)
    plt.title("Ward dendrogram on selected SAX features")
    plt.tight_layout()
    plt.savefig(outpath, dpi=220)
    plt.close()


# Run pipeline

## Data Wrangling

Load one stitched daily Google Trends series per query, audit missingness/zeros, enforce complete daily coverage, and remove near-duplicate query pairs before feature construction.


In [11]:
# -----------------------------------------------------------------------------
# Add S&P 500 volatility as an extra "term" in the clustering panel
# -----------------------------------------------------------------------------

SP500_PATH = Path(r"C:\Python\CSUREMM\shock_detection\SP500_data.csv")
SP500_PRICE_COL = "price"
SP500_VOL_WINDOW = 7

def load_sp500_series(
    path: Path,
    price_col: str = SP500_PRICE_COL,
    vol_window: int = SP500_VOL_WINDOW,
) -> pd.Series:
    """
    Load S&P 500 price and construct rolling realized volatility.

    The CSV contains:
        Time, price, rolling_avg, deviation_pct, shock_up, shock_down, shock

    We use `price` because the other columns are already transformed shock
    diagnostics. The returned volatility series is then added to panel_raw
    and processed by preprocess_panel() with the Google Trends series.
    """
    df = (
        pd.read_csv(path, parse_dates=["Time"])
        .dropna(subset=["Time"])
        .drop_duplicates("Time")
        .sort_values("Time")
        .set_index("Time")
    )

    if price_col not in df.columns:
        raise ValueError(
            f"Column `{price_col}` not found. Available columns: {list(df.columns)}"
        )

    price = df[price_col].astype(float)

    log_return = np.log(price).diff()

    sp500_vol = (
        log_return
        .rolling(
            window=vol_window,
            min_periods=max(3, vol_window // 2)
        )
        .std()
    )

    sp500_vol.name = f"sp500_realized_volatility_{vol_window}d"

    return sp500_vol

In [12]:
series_raw = load_all_series(DATA_DIR)
panel_raw = build_panel(series_raw)

ZERO_SHARE_THRESHOLD = 0.50
EXPECTED_LENGTH = 1612
CORR_THRESHOLD = 0.99

# ---- 0. Initial quality report ----

quality = basic_quality_report(panel_raw)

quality.to_csv(
    OUTPUT_DIR / "01_post_stitch_quality_report.csv",
    index=False
)


# ---- 1. Drop queries with excessive zeros ----

kept_zero_queries = (
    quality
    .loc[quality["zero_share"] <= ZERO_SHARE_THRESHOLD, "query"]
    .tolist()
)

removed_zero_queries = (
    quality
    .loc[quality["zero_share"] > ZERO_SHARE_THRESHOLD, "query"]
    .tolist()
)

zero_filter_df = quality.copy()
zero_filter_df["kept_zero_share"] = (
    zero_filter_df["zero_share"] <= ZERO_SHARE_THRESHOLD
)

zero_filter_df.to_csv(
    OUTPUT_DIR / "01_zero_share_filter.csv",
    index=False
)

panel_raw = panel_raw[kept_zero_queries].copy()

print(f"Removed by zero-share filter: {len(removed_zero_queries)}")
print(f"Remaining after zero-share filter: {panel_raw.shape[1]}")


# ---- 2. Drop queries without exactly 1612 nonmissing observations ----

length_filter_df = pd.DataFrame({
    "query": panel_raw.columns,
    "n_nonmissing": [
        int(panel_raw[col].notna().sum())
        for col in panel_raw.columns
    ],
})

length_filter_df["kept_length"] = (
    length_filter_df["n_nonmissing"] == EXPECTED_LENGTH
)

length_filter_df.to_csv(
    OUTPUT_DIR / "01_length_filter_1612.csv",
    index=False
)

valid_length_queries = (
    length_filter_df
    .loc[length_filter_df["kept_length"], "query"]
    .tolist()
)

invalid_length_queries = (
    length_filter_df
    .loc[~length_filter_df["kept_length"], "query"]
    .tolist()
)

panel_raw = panel_raw[valid_length_queries].copy()

print(f"Removed by length filter: {len(invalid_length_queries)}")
print(f"Remaining after length filter: {panel_raw.shape[1]}")


# ---- 3. Drop one query from each highly correlated pair ----

corr = panel_raw.corr(
    method="pearson",
    min_periods=EXPECTED_LENGTH
).abs()

upper = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

to_drop = set()
duplicate_pairs = []

for col in upper.columns:
    high_corr_matches = upper.index[upper[col] > CORR_THRESHOLD].tolist()

    for match in high_corr_matches:
        if col not in to_drop and match not in to_drop:
            to_drop.add(match)

            duplicate_pairs.append({
                "kept_query": col,
                "dropped_query": match,
                "correlation": float(upper.loc[match, col]),
            })

corr_duplicate_df = pd.DataFrame(duplicate_pairs)

corr_duplicate_df.to_csv(
    OUTPUT_DIR / "01_high_correlation_dropped_pairs.csv",
    index=False
)

panel_raw = panel_raw.drop(columns=sorted(to_drop)).copy()

print(f"Dropped highly correlated queries: {len(to_drop)}")
print(f"Remaining after correlation filter: {panel_raw.shape[1]}")


# ---- 4. Save final cleaned raw panel and final quality report ----

panel_raw.to_csv(
    OUTPUT_DIR / "01_cleaned_panel_raw.csv"
)

quality_cleaned = basic_quality_report(panel_raw)

quality_cleaned.to_csv(
    OUTPUT_DIR / "01_cleaned_panel_quality_report.csv",
    index=False
)

print("Cleaned output stored as panel_raw")

# -----------------------------------------------------------------------------
# Insert S&P 500 series into panel_raw before preprocessing (removed for clean dataset)
# -----------------------------------------------------------------------------

# sp500_series = load_sp500_series(SP500_PATH)

# sp500_aligned = (
#     sp500_series
#     .reindex(panel_raw.index)
#     .interpolate(limit_direction="both")
# )

# panel_raw[sp500_series.name] = sp500_aligned

# print(
#     f"Added {sp500_series.name} to panel_raw: "
#     f"{panel_raw[sp500_series.name].notna().sum()} non-missing observations"
# )

Loaded 1203 stitched series.
Removed by zero-share filter: 348
Remaining after zero-share filter: 855
Removed by length filter: 8
Remaining after length filter: 847
Dropped highly correlated queries: 17
Remaining after correlation filter: 830
Cleaned output stored as panel_raw


## 3. Preprocessing audit

The cleaned raw panel is interpolated, lightly denoised, detrended, robustly normalized, and softly capped. The S&P 500 series is used only as an external benchmark later, not as a clustering term.


In [13]:
# Keep the external benchmark out of the attention-shape clustering panel.
if "sp500_realized_volatility_7d" in panel_raw.columns:
    panel_raw_for_clustering = panel_raw.drop(columns=["sp500_realized_volatility_7d"]).copy()
else:
    panel_raw_for_clustering = panel_raw.copy()

panel_norm, preprocessing_stages = preprocess_panel(panel_raw_for_clustering)
for name, stage in preprocessing_stages.items():
    stage.to_csv(OUTPUT_DIR / f"02_preprocess_{name}.csv")

panel_norm.to_csv(OUTPUT_DIR / "02_preprocess_normalized_final_clustering_panel.csv")
print(panel_norm.shape)


(1612, 830)


### 3.1 Optional preprocessing diagnostics

Use these plots to inspect whether normalization is producing shape-comparable time series without letting extreme one-off spikes dominate the representation.


In [15]:
comparison = pd.DataFrame({
    "raw_std": preprocessing_stages["detrended"].std(),
    "raw_mad": preprocessing_stages["detrended"].apply(
        lambda s: (s-s.median()).abs().median()
    ),
    "norm_std": preprocessing_stages["normalized"].std(),
})
comparison.describe()

,raw_std,raw_mad,norm_std
count,830.000000,830.000000,830.000000
mean,24.061134,1.830304,9.697036
std,122.735071,1.845031,19.978083
min,1.152317,0.026149,0.999349
25%,4.237118,0.399490,1.765261
50%,6.536440,1.461175,3.644711
75%,11.692424,2.655545,9.836383
max,2835.744269,14.693051,268.180665


In [ ]:
norm_quality = basic_quality_report(panel_norm)
norm_quality.to_csv(OUTPUT_DIR / "02_normalized_quality_report.csv", index=False)

summary_stats = panel_norm.describe().T
summary_stats.to_csv(OUTPUT_DIR / "02_normalized_summary_statistics.csv")
summary_stats.head()


## 4. SAX feature construction and baseline clustering

Tune the sliding SAX representation, build the selected feature matrix, and compare KMeans/Ward candidates.


In [16]:
array_norm = panel_to_array_dict(panel_norm)

settings = make_sax_grid()
print(f"Evaluating {len(settings)} SAX settings...")

tuning_results = tune_sax_representation(array_norm, settings)
tuning_results.to_csv(OUTPUT_DIR / "03_sax_window_step_paa_alphabet_tuning.csv", index=False)

ok_results = tuning_results[tuning_results["status"] == "ok"].copy()
if ok_results.empty:
    raise ValueError("No valid SAX tuning results. Check data length and parameter grid.")

selected = ok_results.iloc[0].to_dict()
selected_params = {
    "window_size": int(selected["window_size"]),
    "step": int(selected["step"]),
    "n_segments": int(selected["n_segments"]),
    "alphabet_size": int(selected["alphabet_size"]),
    "selected_k": int(selected["best_k"]),
    "spike_features": INCLUDE_SPIKE_FEATURES,
}

pd.DataFrame([selected_params]).to_csv(OUTPUT_DIR / "04_selected_sax_parameters.csv", index=False)

print("Selected SAX parameters:")
print(json.dumps(selected_params, indent=2))
print("\nTop 10 SAX settings:")
print(ok_results.head(10).to_string(index=False))

features = build_sax_feature_matrix_fast(
    array_dict=array_norm,
    window_size=selected_params["window_size"],
    n_segments=selected_params["n_segments"],
    alphabet_size=selected_params["alphabet_size"],
    step=selected_params["step"],
    spike_features=selected_params["spike_features"],
)
features.to_csv(OUTPUT_DIR / "05_selected_sax_feature_matrix.csv")

Evaluating 52 SAX settings...
  evaluated 10/52 SAX settings
  evaluated 20/52 SAX settings
  evaluated 30/52 SAX settings
  evaluated 40/52 SAX settings
  evaluated 50/52 SAX settings
Selected SAX parameters:
{
  "window_size": 60,
  "step": 15,
  "n_segments": 16,
  "alphabet_size": 3,
  "selected_k": 2,
  "spike_features": false
}

Top 10 SAX settings:
 window_size  step  n_segments  alphabet_size status  n_terms  n_features  best_k  silhouette  davies_bouldin  calinski_harabasz  min_cluster_frac  norm_entropy
          60    15          16              3     ok      830          19       2    0.331149        1.174146         509.283145          0.355422      0.938817
          60    15           8              3     ok      830          11       2    0.317914        1.299350         414.722689          0.343373      0.928010
          60    15          12              3     ok      830          15       2    0.316533        1.245385         473.839031          0.387952      0.96346

In [17]:
# Final KMeans using the selected k from representation tuning.
labels_kmeans, metrics_kmeans = run_kmeans_final(features, k=selected_params["selected_k"])
cluster_member_table(labels_kmeans).to_csv(OUTPUT_DIR / "06_kmeans_labels.csv", index=False)
cluster_size_table(labels_kmeans).to_csv(OUTPUT_DIR / "06_kmeans_cluster_sizes.csv", index=False)
pd.DataFrame([metrics_kmeans]).to_csv(OUTPUT_DIR / "06_kmeans_metrics.csv", index=False)

# # Hierarchical clustering comparison on the same selected SAX feature matrix. (removed for this run)
# hierarchical_results = run_hierarchical_grid(features, k_grid=K_GRID)
# hierarchical_results.to_csv(OUTPUT_DIR / "07_hierarchical_tuning.csv", index=False)

# best_hier = hierarchical_results[hierarchical_results["status"] == "ok"].iloc[0].to_dict()
# labels_hier, Z_hier, metrics_hier = run_hierarchical_final(
#     features,
#     k=int(best_hier["k"]),
#     method=str(best_hier["method"]),
# )

# cluster_member_table(labels_hier).to_csv(OUTPUT_DIR / "07_hierarchical_labels.csv", index=False)
# cluster_size_table(labels_hier).to_csv(OUTPUT_DIR / "07_hierarchical_cluster_sizes.csv", index=False)
# pd.DataFrame([metrics_hier]).to_csv(OUTPUT_DIR / "07_hierarchical_metrics.csv", index=False)

# Also save Ward with the KMeans-selected k for direct one-to-one comparison.
labels_ward, Z_ward, metrics_ward = run_ward_final(features, k=selected_params["selected_k"])
cluster_member_table(labels_ward).to_csv(OUTPUT_DIR / "07_ward_same_k_labels.csv", index=False)
cluster_size_table(labels_ward).to_csv(OUTPUT_DIR / "07_ward_same_k_cluster_sizes.csv", index=False)
pd.DataFrame([metrics_ward]).to_csv(OUTPUT_DIR / "07_ward_same_k_metrics.csv", index=False)

plot_cluster_average_curves(
    panel_norm=panel_norm,
    labels=labels_kmeans,
    outpath=OUTPUT_DIR / "08_kmeans_cluster_average_curves.png",
    title="KMeans clusters on tuned sliding-SAX features",
)
# plot_cluster_average_curves(
#     panel_norm=panel_norm,
#     labels=labels_hier,
#     outpath=OUTPUT_DIR / "08_hierarchical_cluster_average_curves.png",
#     title="Hierarchical clusters on tuned sliding-SAX features",
# )
# plot_dendrogram_for_features(features, Z_hier, OUTPUT_DIR / "08_hierarchical_dendrogram.png")
plot_dendrogram_for_features(features, Z_ward, OUTPUT_DIR / "08_ward_same_k_dendrogram.png")

print("\nKMeans metrics:")
print(metrics_kmeans)
# print("\nBest hierarchical metrics:")
# print(metrics_hier)
print("\nWard same-k metrics:")
print(metrics_ward)
# print("\nTop hierarchical settings:")
# print(hierarchical_results.head(10).to_string(index=False))
print(f"\nOutputs written to: {OUTPUT_DIR}")


KMeans metrics:
{'k': 2, 'silhouette': 0.33053529262542725, 'davies_bouldin': 1.1731170548623473, 'calinski_harabasz': 509.2872521780366, 'inertia': 9764.2158203125}

Ward same-k metrics:
{'k': 2, 'silhouette': 0.3121638000011444, 'davies_bouldin': 1.21028036473012, 'calinski_harabasz': 446.96421468049436}

Outputs written to: C:\Python\CSUREMM\output\sax_tests_july_08


## Evaluate candidate numbers of clusters for the selected SAX representation

In [18]:
K_CANDIDATES = [2, 3, 4, 5, 6]

k_evaluation = evaluate_clustering_for_features(
    features,
    k_grid=[2, 3, 4, 5, 6],
    silhouette_sample_size=None,
)

k_evaluation.to_csv(
    OUTPUT_DIR / "AA_selected_representation_k_evaluation.csv",
    index=False,
)

print("\nCluster evaluation for selected SAX representation:")
print(
    k_evaluation[
        [
            "k",
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz",
            "min_cluster_frac",
            "norm_entropy",
        ]
    ].to_string(index=False)
)


Cluster evaluation for selected SAX representation:
 k  silhouette  davies_bouldin  calinski_harabasz  min_cluster_frac  norm_entropy
 2    0.330191        1.174146         509.283145          0.355422      0.938817
 3    0.181971        1.832705         340.109103          0.291566      0.990655
 4    0.163348        1.856064         263.898996          0.224096      0.996878
 5    0.141670        1.919165         219.060521          0.090361      0.933883
 6    0.133406        1.891644         194.477038          0.090361      0.958657


## Consensus clustering on the selected representation

Runs on top of the `features` matrix built in the cell above, using the
SAX setting chosen by the fixed (balance-aware) selection rule.

- **Layered clustering** cuts one Ward tree at `LAYER_KS` to give nested
  coarse/meso/fine cluster columns (`09_layered_clusters.csv`).
- **Consensus clustering** scans `CONSENSUS_K_RANGE`, reports stability
  (`10_consensus_k_scan.csv`), and produces final consensus-based labels
  plus a per-term stability score for the chosen k
  (`11_consensus_labels.csv`).


In [19]:
# --- 6。 Consensus clustering scan across k -----------------------------------
print(f"\nRunning consensus clustering ({CONSENSUS_N_RESAMPLES} resamples per k, "
      f"k in {list(CONSENSUS_K_RANGE)})...")
consensus_scan, consensus_matrices = scan_consensus_k(
    features,
    k_range=CONSENSUS_K_RANGE,
    n_resamples=CONSENSUS_N_RESAMPLES,
    subsample_frac=CONSENSUS_SUBSAMPLE_FRAC,
)
consensus_scan.to_csv(OUTPUT_DIR / "10_consensus_k_scan.csv", index=False)
print(consensus_scan.to_string(index=False))

# Pick k with the highest consensus CDF area whose gain over the previous k
# has already leveled off (classic delta-K rule), rather than always taking
# the single highest silhouette k as before.
stable_candidates = consensus_scan[consensus_scan["delta_area"].abs() < 0.02]
if not stable_candidates.empty:
    consensus_k = int(stable_candidates.iloc[0]["k"])
else:
    consensus_k = int(consensus_scan.loc[consensus_scan["consensus_cdf_area"].idxmax(), "k"])
print(f"\nSelected consensus k = {consensus_k}")

C_selected = consensus_matrices[consensus_k]
labels_consensus = consensus_labels(C_selected, consensus_k, features.index)
stability = consensus_term_stability(C_selected, labels_consensus)

consensus_table = (
    cluster_member_table(labels_consensus)
    .merge(stability.rename("stability_score").reset_index().rename(columns={"index": "term"}),
           on="term", how="left")
    .sort_values(["cluster", "stability_score"], ascending=[True, False])
    .reset_index(drop=True)
)
consensus_table.to_csv(OUTPUT_DIR / "11_consensus_labels.csv", index=False)
cluster_size_table(labels_consensus).to_csv(OUTPUT_DIR / "11_consensus_cluster_sizes.csv", index=False)

print(f"\nConsensus cluster sizes (k={consensus_k}):")
print(cluster_size_table(labels_consensus).to_string(index=False))
print("\nLowest-stability (most ambiguous) terms:")
print(consensus_table.sort_values("stability_score").head(10).to_string(index=False))

plot_cluster_average_curves(
    panel_norm=panel_norm,
    labels=labels_consensus,
    outpath=OUTPUT_DIR / "11_consensus_cluster_average_curves.png",
    title=f"Consensus clusters (k={consensus_k}) on tuned sliding-SAX features",
)

print(f"\nConsensus outputs written to: {OUTPUT_DIR}")



Running consensus clustering (200 resamples per k, k in [2, 3, 4, 5, 6, 7, 8, 9, 10])...
 k  consensus_cdf_area  delta_area
 2            0.461630    0.461630
 3            0.664433    0.439318
 4            0.741878    0.116559
 5            0.793365    0.069401
 6            0.824400    0.039118
 7            0.850564    0.031737
 8            0.870950    0.023967
 9            0.883340    0.014226
10            0.894719    0.012881

Selected consensus k = 9

Consensus cluster sizes (k=9):
 cluster  n_terms
       0      107
       1       98
       2      158
       3       98
       4       93
       5       63
       6        5
       7      119
       8       89

Lowest-stability (most ambiguous) terms:
                term  cluster  stability_score
free_covid_test_kits        2        -0.435027
              tiktok        4        -0.277759
           top_gun_2        0        -0.216347
          polls_2024        0        -0.207476
         youtube.com        7        -0.17208

In [20]:
# ------------------------------------------------------------------
# Plot the normalized time series of the top 20 most stable terms
# in each consensus cluster
# ------------------------------------------------------------------

import matplotlib.pyplot as plt

TOP_N = 20

# Use the normalized panel from the preprocessing pipeline
panel_plot = preprocessing_stages["normalized"]

for cluster_id in sorted(consensus_table["cluster"].unique()):

    top_terms = (
        consensus_table.loc[
            consensus_table["cluster"] == cluster_id,
            ["term", "stability_score"]
        ]
        .head(TOP_N)["term"]
        .tolist()
    )

    plt.figure(figsize=(14, 5))

    for term in top_terms:
        if term in panel_plot.columns:
            plt.plot(
                panel_plot.index,
                panel_plot[term],
                linewidth=1.2,
                alpha=0.8,
                label=term,
            )

    plt.title(f"Cluster {cluster_id}: Top {TOP_N} Most Stable Words (Normalized)")
    plt.xlabel("Time")
    plt.ylabel("Normalized Search Interest")
    plt.grid(alpha=0.3)
    plt.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        fontsize=9,
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / f"cluster_{cluster_id}_top{TOP_N}_normalized.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

## 5. Internal validation and cluster geometry

These checks summarize within-cluster similarity, between-cluster separation, and the stability of the final consensus partition.


# Internal Validation

In [21]:
# ------------------------------------------------------------------
# Internal validation of consensus clusters
# ------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity


# ------------------------------------------------------------------
# Inputs expected from previous cells:
#   consensus_table: columns = term, cluster, stability_score
#   labels_consensus: pd.Series or dict-like, index/keys = term
#   features: selected SAX feature matrix, index = term
#   panel_norm: normalized time-series panel, columns = term
#   OUTPUT_DIR
# ------------------------------------------------------------------

VALIDATION_DIR = OUTPUT_DIR / "consensus_cluster_internal_validation"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Internal validation of consensus clusters
# Focus: SAX-space cohesion, centroid similarity, stability,
# and descriptive cluster trajectories.
# ------------------------------------------------------------------

def average_pairwise_cosine_similarity(X: np.ndarray) -> float:
    if X.shape[0] < 2:
        return np.nan

    sim = cosine_similarity(X)
    upper = sim[np.triu_indices_from(sim, k=1)]

    return float(np.mean(upper))


def centroid_similarity_summary(X: np.ndarray) -> dict:
    if X.shape[0] < 2:
        return {
            "mean_centroid_dist": np.nan,
            "median_centroid_dist": np.nan,
            "max_centroid_dist": np.nan,
            "mean_centroid_cosine": np.nan,
            "median_centroid_cosine": np.nan,
            "min_centroid_cosine": np.nan,
        }

    centroid = X.mean(axis=0, keepdims=True)

    dist = np.linalg.norm(X - centroid, axis=1)

    cos = cosine_similarity(X, centroid).ravel()

    return {
        "mean_centroid_dist": float(np.mean(dist)),
        "median_centroid_dist": float(np.median(dist)),
        "max_centroid_dist": float(np.max(dist)),
        "mean_centroid_cosine": float(np.mean(cos)),
        "median_centroid_cosine": float(np.median(cos)),
        "min_centroid_cosine": float(np.min(cos)),
    }

In [22]:
# ------------------------------------------------------------------
# Align terms across consensus labels, SAX features, and time-series panel
# ------------------------------------------------------------------

common_terms = (
    consensus_table["term"]
    .loc[consensus_table["term"].isin(features.index)]
    .loc[lambda s: s.isin(panel_norm.columns)]
    .unique()
)

labels_df = (
    consensus_table
    .loc[consensus_table["term"].isin(common_terms)]
    [["term", "cluster", "stability_score"]]
    .copy()
)

features_valid = features.loc[common_terms].copy()

X_scaled = pd.DataFrame(
    StandardScaler().fit_transform(features_valid),
    index=features_valid.index,
    columns=features_valid.columns,
)

panel_valid = panel_norm.loc[:, common_terms].copy()

In [23]:
# ------------------------------------------------------------------
# Cluster-level validation summary
# ------------------------------------------------------------------

validation_rows = []
term_centroid_rows = []

for cluster_id in sorted(labels_df["cluster"].unique()):

    terms = (
        labels_df
        .loc[labels_df["cluster"] == cluster_id, "term"]
        .tolist()
    )

    X_cluster_df = X_scaled.loc[terms]
    X_cluster = X_cluster_df.to_numpy()

    stability_cluster = (
        labels_df
        .loc[labels_df["cluster"] == cluster_id, "stability_score"]
        .dropna()
    )

    q25 = stability_cluster.quantile(0.25)
    q75 = stability_cluster.quantile(0.75)

    centroid_summary = centroid_similarity_summary(X_cluster)

    row = {
        "cluster": cluster_id,
        "n_terms": len(terms),

        # SAX-space cohesion
        "avg_sax_cosine_similarity": average_pairwise_cosine_similarity(X_cluster),

        # Compactness / centroid similarity in clustered feature space
        **centroid_summary,

        # Consensus stability distribution
        "stability_min": float(stability_cluster.min()),
        "stability_q25": float(q25),
        "stability_median": float(stability_cluster.median()),
        "stability_q75": float(q75),
        "stability_iqr": float(q75 - q25),
        "stability_mean": float(stability_cluster.mean()),
    }

    validation_rows.append(row)

    # Term-level distance/similarity to cluster centroid
    centroid = X_cluster_df.mean(axis=0).to_numpy().reshape(1, -1)

    term_dist = np.linalg.norm(
        X_cluster_df.to_numpy() - centroid,
        axis=1,
    )

    term_cos = cosine_similarity(
        X_cluster_df.to_numpy(),
        centroid,
    ).ravel()

    temp = pd.DataFrame({
        "term": terms,
        "cluster": cluster_id,
        "distance_to_cluster_centroid": term_dist,
        "cosine_to_cluster_centroid": term_cos,
    })

    term_centroid_rows.append(temp)


cluster_validation = pd.DataFrame(validation_rows)

cluster_validation.to_csv(
    VALIDATION_DIR / "cluster_internal_validation_summary.csv",
    index=False,
)

print("\nCluster internal validation summary:")
print(cluster_validation.to_string(index=False))


term_centroid_validation = (
    pd.concat(term_centroid_rows, ignore_index=True)
    .merge(
        labels_df[["term", "stability_score"]],
        on="term",
        how="left",
    )
    .sort_values(["cluster", "cosine_to_cluster_centroid"], ascending=[True, False])
)

term_centroid_validation.to_csv(
    VALIDATION_DIR / "term_centroid_validation.csv",
    index=False,
)


Cluster internal validation summary:
 cluster  n_terms  avg_sax_cosine_similarity  mean_centroid_dist  median_centroid_dist  max_centroid_dist  mean_centroid_cosine  median_centroid_cosine  min_centroid_cosine  stability_min  stability_q25  stability_median  stability_q75  stability_iqr  stability_mean
       0      107                   0.320560            2.772324              2.697633           4.852943              0.570902                0.574498             0.119726      -0.216347       0.379256          0.641681       0.754187       0.374931        0.551788
       1       98                   0.475081            2.618028              2.510865           6.386147              0.692841                0.701618             0.298693       0.008494       0.470422          0.644006       0.730129       0.259707        0.578894
       2      158                   0.253402            2.643997              2.560550           4.701973              0.507554                0.524527          

In [24]:
# ------------------------------------------------------------------
# Cluster interpretation figures:
#   1. SAX segment centroid profiles
#   2. Average symbol-frequency profiles
#   3. Top 10 prototype queries by centroid cosine similarity
# ------------------------------------------------------------------

# ------------------------------------------------------------------
# Prepare aligned feature + label data
# ------------------------------------------------------------------

labels_df = (
    consensus_table[["term", "cluster", "stability_score"]]
    .loc[consensus_table["term"].isin(features.index)]
    .copy()
)

features_labeled = features.loc[labels_df["term"]].copy()
features_labeled["term"] = features_labeled.index
features_labeled = features_labeled.merge(
    labels_df,
    on="term",
    how="left",
)


# ------------------------------------------------------------------
# Identify SAX feature groups
# ------------------------------------------------------------------

seg_cols = [
    c for c in features.columns
    if c.startswith("sax_seg_mean_")
]

symbol_cols = [
    c for c in features.columns
    if c.startswith("symbol_share_")
]


# ------------------------------------------------------------------
# Option 1: plot cluster centroid profiles across SAX/PAA segments
# ------------------------------------------------------------------

segment_profiles = (
    features_labeled
    .groupby("cluster")[seg_cols]
    .mean()
)

segment_profiles.to_csv(
    VALIDATION_DIR / "cluster_sax_segment_centroid_profiles.csv"
)

for cluster_id, row in segment_profiles.iterrows():

    x = np.arange(1, len(seg_cols) + 1)

    plt.figure(figsize=(8, 4))

    plt.plot(
        x,
        row.values,
        marker="o",
        linewidth=2,
    )

    plt.xticks(x)
    plt.xlabel("PAA segment position within SAX window")
    plt.ylabel("Average symbolic level")
    plt.title(f"Cluster {cluster_id}: SAX Segment Centroid Profile")
    plt.grid(alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        VALIDATION_DIR / f"cluster_{cluster_id}_sax_segment_centroid.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()




In [25]:
# ------------------------------------------------------------------
# Option 2: plot average symbol-frequency profiles
# ------------------------------------------------------------------

symbol_profiles = (
    features_labeled
    .groupby("cluster")[symbol_cols]
    .mean()
)

symbol_profiles.to_csv(
    VALIDATION_DIR / "cluster_symbol_share_profiles.csv"
)

for cluster_id, row in symbol_profiles.iterrows():

    symbol_ids = [
        int(c.replace("symbol_share_", ""))
        for c in symbol_cols
    ]

    plt.figure(figsize=(6, 4))

    plt.bar(
        symbol_ids,
        row.values,
    )

    plt.xticks(symbol_ids)
    plt.xlabel("SAX symbol")
    plt.ylabel("Average share")
    plt.title(f"Cluster {cluster_id}: Average SAX Symbol Distribution")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        VALIDATION_DIR / f"cluster_{cluster_id}_symbol_share_profile.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()


In [26]:
# ------------------------------------------------------------------
# Option 3: top 10 prototype queries by centroid cosine similarity
# ------------------------------------------------------------------

# Recompute centroid cosine if term_centroid_validation is not already available.
if "term_centroid_validation" not in globals():

    X_scaled = pd.DataFrame(
        StandardScaler().fit_transform(features),
        index=features.index,
        columns=features.columns,
    )

    proto_rows = []

    for cluster_id in sorted(labels_df["cluster"].unique()):

        terms = (
            labels_df
            .loc[labels_df["cluster"] == cluster_id, "term"]
            .tolist()
        )

        X_cluster = X_scaled.loc[terms]
        centroid = X_cluster.mean(axis=0).to_numpy().reshape(1, -1)

        centroid_cos = cosine_similarity(
            X_cluster.to_numpy(),
            centroid,
        ).ravel()

        temp = pd.DataFrame({
            "term": terms,
            "cluster": cluster_id,
            "cosine_to_cluster_centroid": centroid_cos,
        })

        proto_rows.append(temp)

    term_centroid_validation = (
        pd.concat(proto_rows, ignore_index=True)
        .merge(
            labels_df[["term", "stability_score"]],
            on="term",
            how="left",
        )
    )


top10_prototypes = (
    term_centroid_validation
    .sort_values(
        ["cluster", "cosine_to_cluster_centroid"],
        ascending=[True, False],
    )
    .groupby("cluster")
    .head(10)
    .reset_index(drop=True)
)

top10_prototypes.to_csv(
    VALIDATION_DIR / "top10_prototype_queries_by_cluster.csv",
    index=False,
)

print("\nTop 10 prototype queries by cluster:")
print(
    top10_prototypes[
        [
            "cluster",
            "term",
            "cosine_to_cluster_centroid",
            "stability_score",
        ]
    ].to_string(index=False)
)


Top 10 prototype queries by cluster:
 cluster                         term  cosine_to_cluster_centroid  stability_score
       0                   tsla_stock                    0.921094         0.767813
       0                        sonic                    0.882401         0.764741
       0                         tsla                    0.870285         0.773282
       0                   ny_yankees                    0.865847         0.777012
       0               world_cup_2022                    0.862453         0.623656
       0               andrew_wiggins                    0.853833         0.770011
       0        golden_state_warriors                    0.848439         0.742170
       0                dennis_rodman                    0.834735         0.742766
       0            tesla_stock_price                    0.820083         0.766168
       0                     warriors                    0.817716         0.762242
       1             charles_oliveira            

## 6. External validation with S&P 500 volatility

Market volatility is kept outside the clustering features and used only as an outcome/benchmark.


# S&P 500

In [27]:
# -----------------------------------------------------------------------------
# External S&P 500 benchmark + cluster index analysis
# Do NOT add S&P 500 to the clustering panel.
# Use it after clustering as an external outcome/benchmark.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error


# -----------------------------------------------------------------------------
# Load S&P 500 price, returns, and realized volatility
# -----------------------------------------------------------------------------

SP500_PATH = Path(r"C:\Python\CSUREMM\shock_detection\SP500_data.csv")
SP500_PRICE_COL = "price"
SP500_VOL_WINDOW = 7

MARKET_DIR = OUTPUT_DIR / "consensus_cluster_sp500_analysis"
MARKET_DIR.mkdir(parents=True, exist_ok=True)


def load_sp500_market_panel(
    path: Path,
    price_col: str = SP500_PRICE_COL,
    vol_window: int = SP500_VOL_WINDOW,
) -> pd.DataFrame:

    df = (
        pd.read_csv(path, parse_dates=["Time"])
        .dropna(subset=["Time"])
        .drop_duplicates("Time")
        .sort_values("Time")
        .set_index("Time")
    )

    if price_col not in df.columns:
        raise ValueError(
            f"Column `{price_col}` not found. Available columns: {list(df.columns)}"
        )

    price = df[price_col].astype(float)

    log_return = np.log(price).diff()

    realized_vol = (
        log_return
        .rolling(
            window=vol_window,
            min_periods=max(3, vol_window // 2),
        )
        .std()
    )

    market = pd.DataFrame({
        "sp500_price": price,
        "sp500_log_return": log_return,
        f"sp500_realized_volatility_{vol_window}d": realized_vol,
    })

    return market


market = load_sp500_market_panel(SP500_PATH)


# -----------------------------------------------------------------------------
# Construct one Google Trends index per consensus cluster
# Uses normalized Google Trends series only.
# S&P 500 should NOT be included as a cluster member.
# -----------------------------------------------------------------------------

def build_cluster_indices(
    panel_norm: pd.DataFrame,
    consensus_table: pd.DataFrame,
    weighting: str = "stability",
    top_n: int | None = None,
) -> pd.DataFrame:

    index_dict = {}

    for cluster_id in sorted(consensus_table["cluster"].unique()):

        cluster_terms_df = (
            consensus_table
            .loc[consensus_table["cluster"] == cluster_id]
            .copy()
        )

        cluster_terms_df = cluster_terms_df[
            cluster_terms_df["term"].isin(panel_norm.columns)
        ]

        if top_n is not None:
            cluster_terms_df = (
                cluster_terms_df
                .sort_values("stability_score", ascending=False)
                .head(top_n)
            )

        terms = cluster_terms_df["term"].tolist()

        if len(terms) == 0:
            continue

        X = panel_norm[terms].copy()

        if weighting == "equal":
            index_series = X.mean(axis=1)

        elif weighting == "stability":
            w = cluster_terms_df["stability_score"].astype(float).copy()

            # Shift weights to be nonnegative within cluster
            w = w - w.min()

            if w.sum() == 0:
                w = pd.Series(np.ones(len(w)), index=w.index)

            w = w / w.sum()

            index_series = X.mul(w.to_numpy(), axis=1).sum(axis=1)

        else:
            raise ValueError("weighting must be either 'equal' or 'stability'")

        index_series.name = f"cluster_{cluster_id}_index"
        index_dict[index_series.name] = index_series

    return pd.DataFrame(index_dict)


cluster_indices_equal = build_cluster_indices(
    panel_norm=panel_norm,
    consensus_table=consensus_table,
    weighting="equal",
    top_n=None,
)

cluster_indices_stability = build_cluster_indices(
    panel_norm=panel_norm,
    consensus_table=consensus_table,
    weighting="stability",
    top_n=None,
)

cluster_indices_top20 = build_cluster_indices(
    panel_norm=panel_norm,
    consensus_table=consensus_table,
    weighting="stability",
    top_n=20,
)

cluster_indices_equal.to_csv(MARKET_DIR / "cluster_indices_equal_weight.csv")
cluster_indices_stability.to_csv(MARKET_DIR / "cluster_indices_stability_weighted.csv")
cluster_indices_top20.to_csv(MARKET_DIR / "cluster_indices_top20_stability_weighted.csv")

In [28]:
# -----------------------------------------------------------------------------
# Align cluster indices with S&P 500 market data
# -----------------------------------------------------------------------------

target_col = f"sp500_realized_volatility_{SP500_VOL_WINDOW}d"

analysis_df = (
    cluster_indices_stability
    .join(market, how="inner")
    .dropna()
)

analysis_df.to_csv(MARKET_DIR / "cluster_indices_with_sp500.csv")


# -----------------------------------------------------------------------------
# Shape comparison: cluster indices vs S&P realized volatility
# Normalize each series for visual comparison only.
# -----------------------------------------------------------------------------

def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std()


plot_df = analysis_df.copy()

for col in cluster_indices_stability.columns:
    plot_df[col] = zscore(plot_df[col])

plot_df[target_col] = zscore(plot_df[target_col])


for col in cluster_indices_stability.columns:

    plt.figure(figsize=(14, 5))

    plt.plot(
        plot_df.index,
        plot_df[col],
        label=col,
        linewidth=1.5,
    )

    plt.plot(
        plot_df.index,
        plot_df[target_col],
        label=target_col,
        linewidth=1.5,
        alpha=0.8,
    )

    plt.axhline(0, linewidth=1, alpha=0.5)

    plt.title(f"Shape Comparison: {col} vs S&P 500 Realized Volatility")
    plt.xlabel("Time")
    plt.ylabel("Z-scored value")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        MARKET_DIR / f"shape_comparison_{col}_vs_sp500_vol.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()


# -----------------------------------------------------------------------------
# Lead-lag correlation test
# Corr(cluster_index_t, market_target_{t+h})
# h > 0 means the cluster index leads the S&P variable.
# -----------------------------------------------------------------------------

def lead_lag_correlation(
    df: pd.DataFrame,
    index_cols: list[str],
    target_col: str,
    max_lag: int = 30,
) -> pd.DataFrame:

    rows = []

    for index_col in index_cols:

        for h in range(-max_lag, max_lag + 1):

            corr = df[index_col].corr(
                df[target_col].shift(-h)
            )

            rows.append({
                "cluster_index": index_col,
                "horizon_days": h,
                "correlation": corr,
            })

    return pd.DataFrame(rows)


lead_lag_corr = lead_lag_correlation(
    analysis_df,
    index_cols=list(cluster_indices_stability.columns),
    target_col=target_col,
    max_lag=30,
)

lead_lag_corr.to_csv(
    MARKET_DIR / "lead_lag_correlations_cluster_indices_vs_sp500_vol.csv",
    index=False,
)


# Plot lead-lag correlation curves
for index_col in cluster_indices_stability.columns:

    temp = lead_lag_corr[
        lead_lag_corr["cluster_index"] == index_col
    ]

    plt.figure(figsize=(8, 4))

    plt.plot(
        temp["horizon_days"],
        temp["correlation"],
        marker="o",
        linewidth=1.5,
    )

    plt.axhline(0, linewidth=1, alpha=0.5)
    plt.axvline(0, linewidth=1, alpha=0.5)

    plt.title(f"Lead-Lag Correlation: {index_col} vs S&P Volatility")
    plt.xlabel("Horizon h: corr(index_t, market_{t+h})")
    plt.ylabel("Correlation")
    plt.grid(alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        MARKET_DIR / f"lead_lag_corr_{index_col}_vs_sp500_vol.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()


# -----------------------------------------------------------------------------
# Predictive regressions:
# target_{t+h} = alpha + beta * cluster_index_t
# -----------------------------------------------------------------------------

def predictive_regression_scan(
    df: pd.DataFrame,
    index_cols: list[str],
    target_col: str,
    horizons: list[int],
) -> pd.DataFrame:

    rows = []

    for index_col in index_cols:

        for h in horizons:

            y = df[target_col].shift(-h)
            X = df[[index_col]]

            reg_df = (
                pd.concat([X, y.rename("target")], axis=1)
                .dropna()
            )

            if len(reg_df) < 30:
                continue

            X_mat = reg_df[[index_col]].to_numpy()
            y_vec = reg_df["target"].to_numpy()

            model = LinearRegression()
            model.fit(X_mat, y_vec)

            y_hat = model.predict(X_mat)

            rows.append({
                "cluster_index": index_col,
                "target": target_col,
                "horizon_days": h,
                "n_obs": len(reg_df),
                "beta": float(model.coef_[0]),
                "intercept": float(model.intercept_),
                "r2": float(r2_score(y_vec, y_hat)),
                "rmse": float(mean_squared_error(y_vec, y_hat) ** 0.5),
                "corr": float(pd.Series(reg_df[index_col]).corr(reg_df["target"])),
            })

    return pd.DataFrame(rows)


prediction_results = predictive_regression_scan(
    analysis_df,
    index_cols=list(cluster_indices_stability.columns),
    target_col=target_col,
    horizons=[1, 7, 14, 30, 60],
)

prediction_results.to_csv(
    MARKET_DIR / "predictive_regressions_cluster_indices_vs_sp500_vol.csv",
    index=False,
)

print("\nTop predictive cluster-index results:")
print(
    prediction_results
    .sort_values(["r2", "corr"], ascending=[False, False])
    .head(20)
    .to_string(index=False)
)


Top predictive cluster-index results:
  cluster_index                       target  horizon_days  n_obs      beta  intercept       r2     rmse      corr
cluster_4_index sp500_realized_volatility_7d             7   1095  0.000778   0.006994 0.142516 0.004128  0.377513
cluster_2_index sp500_realized_volatility_7d             7   1095  0.000651   0.007193 0.130756 0.004156  0.361602
cluster_2_index sp500_realized_volatility_7d             1   1101  0.000631   0.007223 0.123061 0.004168  0.350800
cluster_4_index sp500_realized_volatility_7d             1   1101  0.000708   0.007122 0.118139 0.004180  0.343714
cluster_0_index sp500_realized_volatility_7d             1   1101  0.000619   0.007612 0.098099 0.004227  0.313208
cluster_0_index sp500_realized_volatility_7d             7   1095  0.000605   0.007644 0.093689 0.004244  0.306087
cluster_4_index sp500_realized_volatility_7d            14   1088  0.000534   0.007475 0.067025 0.004316  0.258892
cluster_2_index sp500_realized_volatility

In [ ]:
# -----------------------------------------------------------------------------
# Optional: compare index construction choices
# equal-weight vs stability-weight vs top-20 stability-weight
# -----------------------------------------------------------------------------

index_sets = {
    "equal": cluster_indices_equal,
    "stability": cluster_indices_stability,
    "top20_stability": cluster_indices_top20,
}

all_prediction_results = []

for index_name, index_panel in index_sets.items():

    temp_df = (
        index_panel
        .join(market, how="inner")
        .dropna()
    )

    temp_results = predictive_regression_scan(
        temp_df,
        index_cols=list(index_panel.columns),
        target_col=target_col,
        horizons=[1, 7, 14, 30, 60],
    )

    temp_results["index_construction"] = index_name

    all_prediction_results.append(temp_results)


all_prediction_results = pd.concat(
    all_prediction_results,
    ignore_index=True,
)

all_prediction_results.to_csv(
    MARKET_DIR / "predictive_regressions_all_index_constructions.csv",
    index=False,
)

print("\nBest results across index-construction methods:")
print(
    all_prediction_results
    .sort_values(["r2", "corr"], ascending=[False, False])
    .head(30)
    .to_string(index=False)
)

print(f"\nMarket comparison and prediction outputs written to: {MARKET_DIR}")

In [ ]:
# Inspect Cluster 2 terms
print(
    consensus_table
    .loc[consensus_table["cluster"] == 2]
    .sort_values("stability_score", ascending=False)
    .head(40)
    .to_string(index=False)
)

In [ ]:
# ------------------------------------------------------------
# Out-of-sample test with train-only scaling + expanding window
# ------------------------------------------------------------

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import pandas as pd

target_col = "sp500_realized_volatility_7d"
index_col = "cluster_2_index"
h = 1

df = analysis_df[[index_col, target_col]].copy()
df["target_future"] = df[target_col].shift(-h)
df = df.dropna()

initial_train_frac = 0.70
initial_train_size = int(len(df) * initial_train_frac)

pred_rows = []

for t in range(initial_train_size, len(df)):

    train = df.iloc[:t]
    test = df.iloc[[t]]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train[[index_col]])
    X_test = scaler.transform(test[[index_col]])

    y_train = train["target_future"]
    y_test = test["target_future"].iloc[0]

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)[0]

    pred_rows.append({
        "date": test.index[0],
        "actual": y_test,
        "predicted": y_pred,
        "beta": model.coef_[0],
        "intercept": model.intercept_,
    })

oos_pred = pd.DataFrame(pred_rows).set_index("date")

test_r2 = r2_score(oos_pred["actual"], oos_pred["predicted"])
test_rmse = mean_squared_error(oos_pred["actual"], oos_pred["predicted"]) ** 0.5
test_corr = oos_pred["predicted"].corr(oos_pred["actual"])

print("Expanding-window out-of-sample test")
print("Cluster:", index_col)
print("Horizon:", h)
print("Initial train n:", initial_train_size)
print("Test n:", len(oos_pred))
print("OOS R2:", test_r2)
print("OOS RMSE:", test_rmse)
print("OOS corr:", test_corr)

In [ ]:
# ------------------------------------------------------------------
# External validation: OOS prediction tests for all cluster indices
# ------------------------------------------------------------------

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------------
# Expanding-window prediction helper
# ------------------------------------------------------------------

def expanding_window_oos(
    df: pd.DataFrame,
    predictors: list[str],
    target_col: str,
    horizon: int = 1,
    initial_train_frac: float = 0.70,
) -> tuple[pd.DataFrame, dict]:

    # remove accidental duplicate columns
    cols = list(dict.fromkeys(predictors + [target_col]))

    work = df[cols].copy()

    # force target to be a Series even if duplicates existed upstream
    y = df[target_col]
    if isinstance(y, pd.DataFrame):
        y = y.iloc[:, 0]

    work["target_future"] = y.shift(-horizon)
    work = work.dropna()

    initial_train_size = int(len(work) * initial_train_frac)

    preds = []

    for i in range(initial_train_size, len(work)):
        train = work.iloc[:i]
        test = work.iloc[[i]]

        X_train = train[predictors]
        y_train = train["target_future"]

        X_test = test[predictors]

        model = LinearRegression()
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)[0]

        preds.append({
            "date": test.index[0],
            "actual": test["target_future"].iloc[0],
            "predicted": y_pred,
        })

    pred_df = pd.DataFrame(preds).set_index("date")

    metrics = {
        "oos_rmse": np.sqrt(mean_squared_error(pred_df["actual"], pred_df["predicted"])),
        "oos_corr": pred_df["actual"].corr(pred_df["predicted"]),
        "test_n": len(pred_df),
        "initial_train_n": initial_train_size,
    }

    return pred_df, metrics



OOS results: cluster-only models


KeyError: 'horizon_days'

In [ ]:
# ------------------------------------------------------------------
# Test 1: run same OOS test for every cluster index
# ------------------------------------------------------------------

target_col = "sp500_realized_volatility_7d"
cluster_index_cols = [
    c for c in analysis_df.columns
    if c.startswith("cluster_") and c.endswith("_index")
]

horizons = [1, 7, 14, 30]

oos_metrics = []
oos_predictions = {}

for h in horizons:

    for cluster_col in cluster_index_cols:

        pred, metrics = expanding_window_oos(
            df=analysis_df,
            predictors=[cluster_col],
            target_col=target_col,
            horizon=h,
        )

        metrics["horizon_days"] = h
        metrics["predictors"] = cluster_col
        metrics["cluster_index"] = cluster_col
        metrics["model"] = "cluster_only"

        oos_metrics.append(metrics)
        oos_predictions[(h, cluster_col)] = pred

oos_metrics = pd.DataFrame(oos_metrics)

oos_metrics.to_csv(
    MARKET_DIR / "oos_all_clusters_cluster_only.csv",
    index=False,
)

print("\nOOS results: cluster-only models")
print(
    oos_metrics
    .sort_values(["horizon_days", "oos_r2"], ascending=[True, False])
    .to_string(index=False)
)


## Tests for Cluster 2 only

In [30]:
# ------------------------------------------------------------------
# Clinic brief tests: Cluster 2 external validation
# ------------------------------------------------------------------

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CLINIC_DIR = MARKET_DIR / "clinic_brief_cluster2"
CLINIC_DIR.mkdir(parents=True, exist_ok=True)

target_col = "sp500_realized_volatility_7d"
cluster_id = 2
horizons = [1, 7]
cluster_col = f"cluster_{cluster_id}_index"


# ------------------------------------------------------------------
# Helper: fixed split OOS regression
# ------------------------------------------------------------------

def fixed_split_oos(
    df,
    predictors,
    target_col,
    horizon=1,
    train_frac=0.70,
):
    cols = list(dict.fromkeys(predictors + [target_col]))

    work = df.loc[:, cols].copy()

    # if duplicate column names exist, keep the first occurrence
    work = work.loc[:, ~work.columns.duplicated()]

    target_series = work[target_col].squeeze()
    work["target_future"] = target_series.shift(-horizon)

    work = work.dropna()

    split = int(len(work) * train_frac)

    train = work.iloc[:split]
    test = work.iloc[split:]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train[predictors])
    X_test = scaler.transform(test[predictors])

    y_train = train["target_future"]
    y_test = test["target_future"]

    model = LinearRegression()
    model.fit(X_train, y_train)

    pred = pd.Series(
        model.predict(X_test),
        index=test.index,
        name="predicted",
    )

    metrics = {
        "predictors": "+".join(predictors),
        "horizon_days": horizon,
        "train_n": len(train),
        "test_n": len(test),
        "oos_r2": r2_score(y_test, pred),
        "oos_rmse": mean_squared_error(y_test, pred) ** 0.5,
        "oos_corr": pred.corr(y_test),
    }

    pred_df = pd.DataFrame({
        "actual": y_test,
        "predicted": pred,
    })

    return pred_df, metrics

In [31]:
# ------------------------------------------------------------------
# 1. Incremental benchmark test:
#    Vol_{t+h} ~ Vol_t
#    vs.
#    Vol_{t+h} ~ Vol_t + Cluster 2 index
# ------------------------------------------------------------------

incremental_rows = []
prediction_store = {}

for h in horizons:

    pred_bench, m_bench = fixed_split_oos(
        df=analysis_df,
        predictors=[target_col],
        target_col=target_col,
        horizon=h,
    )

    m_bench["model"] = "vol_persistence"

    pred_cluster, m_cluster = fixed_split_oos(
        df=analysis_df,
        predictors=[cluster_col],
        target_col=target_col,
        horizon=h,
    )

    m_cluster["model"] = "cluster2_only"

    pred_incremental, m_incremental = fixed_split_oos(
        df=analysis_df,
        predictors=[target_col, cluster_col],
        target_col=target_col,
        horizon=h,
    )

    m_incremental["model"] = "vol_persistence_plus_cluster2"

    prediction_store[(h, "benchmark")] = pred_bench
    prediction_store[(h, "cluster2_only")] = pred_cluster
    prediction_store[(h, "incremental")] = pred_incremental

    incremental_rows.extend([m_bench, m_cluster, m_incremental])

clinic_oos = pd.DataFrame(incremental_rows)

benchmark_r2 = (
    clinic_oos
    .loc[clinic_oos["model"] == "vol_persistence", ["horizon_days", "oos_r2", "oos_rmse"]]
    .rename(columns={
        "oos_r2": "benchmark_oos_r2",
        "oos_rmse": "benchmark_oos_rmse",
    })
)

clinic_oos = clinic_oos.merge(
    benchmark_r2,
    on="horizon_days",
    how="left",
)

clinic_oos["incremental_oos_r2"] = (
    clinic_oos["oos_r2"] - clinic_oos["benchmark_oos_r2"]
)

clinic_oos["rmse_improvement"] = (
    clinic_oos["benchmark_oos_rmse"] - clinic_oos["oos_rmse"]
)

clinic_oos.to_csv(
    CLINIC_DIR / "cluster2_incremental_oos_tests.csv",
    index=False,
)

print("\nCluster 2 incremental OOS tests:")
print(clinic_oos.to_string(index=False))



Cluster 2 incremental OOS tests:
                                  predictors  horizon_days  train_n  test_n   oos_r2  oos_rmse  oos_corr                         model  benchmark_oos_r2  benchmark_oos_rmse  incremental_oos_r2  rmse_improvement
                sp500_realized_volatility_7d             1      770     331 0.862220  0.002039  0.928737               vol_persistence          0.862220            0.002039            0.000000          0.000000
                             cluster_2_index             1      770     331 0.137226  0.005103  0.543098                 cluster2_only          0.862220            0.002039           -0.724994         -0.003064
sp500_realized_volatility_7d+cluster_2_index             1      770     331 0.864744  0.002021  0.930163 vol_persistence_plus_cluster2          0.862220            0.002039            0.002524          0.000019
                sp500_realized_volatility_7d             7      766     329 0.133818  0.005122  0.382202               vol

In [32]:
# ------------------------------------------------------------------
# 2. Actual vs predicted plots for Cluster 2 at h = 1 and h = 7
# ------------------------------------------------------------------

for h in horizons:

    pred = prediction_store[(h, "cluster2_only")]

    plt.figure(figsize=(14, 5))

    plt.plot(
        pred.index,
        pred["actual"],
        label="Actual future S&P volatility",
        linewidth=1.5,
    )

    plt.plot(
        pred.index,
        pred["predicted"],
        label="Predicted by Cluster 2 index",
        linewidth=1.5,
    )

    plt.title(f"Cluster 2 OOS Prediction of S&P 500 Volatility, h={h}")
    plt.xlabel("Time")
    plt.ylabel(target_col)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        CLINIC_DIR / f"cluster2_actual_vs_predicted_h{h}.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()

In [33]:
# ------------------------------------------------------------------
# 3. Cluster 2 prototype terms
#    A. by centroid cosine
#    B. by stability score
# ------------------------------------------------------------------

cluster2_by_centroid = (
    term_centroid_validation
    .loc[term_centroid_validation["cluster"] == cluster_id]
    .sort_values("cosine_to_cluster_centroid", ascending=False)
    .head(30)
)

cluster2_by_stability = (
    consensus_table
    .loc[consensus_table["cluster"] == cluster_id]
    .sort_values("stability_score", ascending=False)
    .head(30)
)

cluster2_by_centroid.to_csv(
    CLINIC_DIR / "cluster2_top30_by_centroid_cosine.csv",
    index=False,
)

cluster2_by_stability.to_csv(
    CLINIC_DIR / "cluster2_top30_by_stability.csv",
    index=False,
)

print("\nCluster 2 prototype terms by centroid cosine:")
print(cluster2_by_centroid.to_string(index=False))

print("\nCluster 2 prototype terms by stability:")
print(cluster2_by_stability.to_string(index=False))


# ------------------------------------------------------------------
# 4. Robustness table:
#    Equal-weight vs stability-weight vs top-20 stability-weight
#    Cluster 2 only
# ------------------------------------------------------------------

index_panels = {
    "equal": cluster_indices_equal,
    "stability": cluster_indices_stability,
    "top20_stability": cluster_indices_top20,
}

robustness_rows = []

for construction_name, index_panel in index_panels.items():

    cluster_col = f"cluster_{cluster_id}_index"

    if cluster_col not in index_panel.columns:
        continue

    temp_df = (
        index_panel[[cluster_col]]
        .join(market, how="inner")
        .dropna()
    )

    for h in horizons:

        pred, metrics = fixed_split_oos(
            df=temp_df,
            predictors=[cluster_col],
            target_col=target_col,
            horizon=h,
        )

        metrics["cluster"] = cluster_id
        metrics["index_construction"] = construction_name

        robustness_rows.append(metrics)

cluster2_robustness = pd.DataFrame(robustness_rows)

cluster2_robustness.to_csv(
    CLINIC_DIR / "cluster2_index_construction_robustness.csv",
    index=False,
)

print("\nCluster 2 index-construction robustness:")
print(
    cluster2_robustness
    .sort_values(["horizon_days", "oos_r2"], ascending=[True, False])
    .to_string(index=False)
)

print(f"\nClinic brief outputs written to: {CLINIC_DIR}")


Cluster 2 prototype terms by centroid cosine:
                       term  cluster  distance_to_cluster_centroid  cosine_to_cluster_centroid  stability_score
                 val_kilmer        2                      1.223936                    0.921521         0.260394
           jamie_lee_curtis        2                      2.626091                    0.919688         0.165459
                   nfl_news        2                      1.092282                    0.888022         0.292094
                 telfar_bag        2                      3.291693                    0.876258         0.338933
              yellowjackets        2                      3.934332                    0.870450         0.312481
                kobe_bryant        2                      2.626223                    0.854902        -0.080868
                    packers        2                      4.663166                    0.850137         0.245707
    the_little_mermaid_2023        2                     

## Tests for all clusters

In [34]:
# ------------------------------------------------------------------
# Test 2: benchmark model using volatility persistence only
# Vol_{t+h} ~ Vol_t
# ------------------------------------------------------------------

benchmark_metrics = []
benchmark_predictions = {}

for h in horizons:

    pred, metrics = expanding_window_oos(
        df=analysis_df,
        predictors=[target_col],
        target_col=target_col,
        horizon=h,
        initial_train_frac=0.70,
    )

    metrics["cluster_index"] = "benchmark_vol_persistence"
    metrics["model"] = "vol_persistence"

    benchmark_metrics.append(metrics)
    benchmark_predictions[h] = pred

benchmark_metrics = pd.DataFrame(benchmark_metrics)

benchmark_metrics.to_csv(
    MARKET_DIR / "oos_benchmark_vol_persistence.csv",
    index=False,
)

print("\nOOS benchmark results: volatility persistence")
print(benchmark_metrics.to_string(index=False))

ValueError: Cannot set a DataFrame with multiple columns to the single column target_future

In [35]:
# ------------------------------------------------------------------
# Test 3: incremental model
# Vol_{t+h} ~ Vol_t + ClusterIndex_t
# ------------------------------------------------------------------

incremental_metrics = []
incremental_predictions = {}

for h in horizons:

    for index_col in cluster_index_cols:

        pred, metrics = expanding_window_oos(
            df=analysis_df,
            predictors=[target_col, index_col],
            target_col=target_col,
            horizon=h,
            initial_train_frac=0.70,
        )

        metrics["cluster_index"] = index_col
        metrics["model"] = "vol_persistence_plus_cluster"

        incremental_metrics.append(metrics)
        incremental_predictions[(index_col, h)] = pred

incremental_metrics = pd.DataFrame(incremental_metrics)

incremental_metrics.to_csv(
    MARKET_DIR / "oos_vol_persistence_plus_clusters.csv",
    index=False,
)

print("\nOOS results: volatility persistence + cluster index")
print(
    incremental_metrics
    .sort_values(["horizon_days", "oos_r2"], ascending=[True, False])
    .to_string(index=False)
)

ValueError: Cannot set a DataFrame with multiple columns to the single column target_future

In [36]:
# ------------------------------------------------------------------
# Test 4: incremental OOS R2 over benchmark
# ------------------------------------------------------------------

comparison = incremental_metrics.merge(
    benchmark_metrics[
        ["horizon_days", "oos_r2", "oos_rmse", "oos_corr"]
    ].rename(columns={
        "oos_r2": "benchmark_oos_r2",
        "oos_rmse": "benchmark_oos_rmse",
        "oos_corr": "benchmark_oos_corr",
    }),
    on="horizon_days",
    how="left",
)

comparison["incremental_oos_r2"] = (
    comparison["oos_r2"] - comparison["benchmark_oos_r2"]
)

comparison["rmse_improvement"] = (
    comparison["benchmark_oos_rmse"] - comparison["oos_rmse"]
)

comparison.to_csv(
    MARKET_DIR / "oos_incremental_cluster_value_over_benchmark.csv",
    index=False,
)

print("\nIncremental predictive value over volatility persistence:")
print(
    comparison
    .sort_values(["horizon_days", "incremental_oos_r2"], ascending=[True, False])
    [
        [
            "cluster_index",
            "horizon_days",
            "oos_r2",
            "benchmark_oos_r2",
            "incremental_oos_r2",
            "oos_corr",
            "benchmark_oos_corr",
            "rmse_improvement",
        ]
    ]
    .to_string(index=False)
)


AttributeError: 'list' object has no attribute 'merge'

In [ ]:
# ------------------------------------------------------------------
# Test 5: plot actual vs predicted for best cluster-only model
# ------------------------------------------------------------------

best_row = (
    oos_metrics
    .sort_values("oos_r2", ascending=False)
    .iloc[0]
)

best_cluster = best_row["cluster_index"]
best_h = int(best_row["horizon_days"])

best_pred = all_oos_predictions[(best_cluster, best_h, "cluster_only")]

plt.figure(figsize=(14, 5))

plt.plot(
    best_pred.index,
    best_pred["actual"],
    label="Actual future S&P volatility",
    linewidth=1.5,
)

plt.plot(
    best_pred.index,
    best_pred["predicted"],
    label=f"Predicted by {best_cluster}",
    linewidth=1.5,
)

plt.title(
    f"Out-of-Sample Prediction: {best_cluster}, horizon={best_h} day"
)
plt.xlabel("Time")
plt.ylabel(target_col)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    MARKET_DIR / f"oos_actual_vs_predicted_{best_cluster}_h{best_h}.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ------------------------------------------------------------------
# Test 6: plot incremental model for best cluster over benchmark
# ------------------------------------------------------------------

best_inc_row = (
    comparison
    .sort_values("incremental_oos_r2", ascending=False)
    .iloc[0]
)

best_inc_cluster = best_inc_row["cluster_index"]
best_inc_h = int(best_inc_row["horizon_days"])

best_inc_pred = incremental_predictions[(best_inc_cluster, best_inc_h)]
benchmark_pred = benchmark_predictions[best_inc_h]

plt.figure(figsize=(14, 5))

plt.plot(
    best_inc_pred.index,
    best_inc_pred["actual"],
    label="Actual future S&P volatility",
    linewidth=1.5,
)

plt.plot(
    benchmark_pred.index,
    benchmark_pred["predicted"],
    label="Benchmark: volatility persistence",
    linewidth=1.3,
    alpha=0.8,
)

plt.plot(
    best_inc_pred.index,
    best_inc_pred["predicted"],
    label=f"Benchmark + {best_inc_cluster}",
    linewidth=1.5,
)

plt.title(
    f"Incremental OOS Prediction: {best_inc_cluster}, horizon={best_inc_h} day"
)
plt.xlabel("Time")
plt.ylabel(target_col)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    MARKET_DIR / f"oos_incremental_actual_vs_predicted_{best_inc_cluster}_h{best_inc_h}.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# ------------------------------------------------------------------
# Test 7: inspect prototype terms for best predictive cluster
# ------------------------------------------------------------------

best_cluster_id = int(
    best_cluster
    .replace("cluster_", "")
    .replace("_index", "")
)

print(f"\nTop terms in best predictive cluster {best_cluster_id}:")
print(
    consensus_table
    .loc[consensus_table["cluster"] == best_cluster_id]
    .sort_values("stability_score", ascending=False)
    .head(40)
    .to_string(index=False)
)

print(f"\nOOS testing outputs written to: {MARKET_DIR}")

# 7. Robustness roadmap: verify captured attention shape

The following sections implement the added checks for the clinic brief. They are designed to produce tables and figures that can be cited directly in a report.


## 7.1 Reconcile `k=2` versus `k=9`: nesting test

A clean hierarchy means each fine cluster falls mostly inside one macro cluster. Messy nesting suggests that the fine `k=9` solution is less structurally stable than the macro split.


In [1]:
# ------------------------------------------------------------------
# k=2 vs k=9 nesting test
# ------------------------------------------------------------------
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

NESTING_K_MACRO = 2
NESTING_K_FINE = 9

C_macro = consensus_matrices.get(NESTING_K_MACRO)
if C_macro is None:
    C_macro = consensus_matrix(
        features.to_numpy(dtype=np.float32),
        k=NESTING_K_MACRO,
        n_resamples=CONSENSUS_N_RESAMPLES,
        subsample_frac=CONSENSUS_SUBSAMPLE_FRAC,
        random_state=RANDOM_STATE,
    )

C_fine = consensus_matrices.get(NESTING_K_FINE)
if C_fine is None:
    C_fine = consensus_matrix(
        features.to_numpy(dtype=np.float32),
        k=NESTING_K_FINE,
        n_resamples=CONSENSUS_N_RESAMPLES,
        subsample_frac=CONSENSUS_SUBSAMPLE_FRAC,
        random_state=RANDOM_STATE + NESTING_K_FINE,
    )

labels_k2 = consensus_labels(C_macro, NESTING_K_MACRO, features.index).rename("k2_cluster")
labels_k9 = consensus_labels(C_fine, NESTING_K_FINE, features.index).rename("k9_cluster")

nesting_df = pd.concat([labels_k2, labels_k9], axis=1)
ct_counts = pd.crosstab(nesting_df["k9_cluster"], nesting_df["k2_cluster"])
ct_share = ct_counts.div(ct_counts.sum(axis=1), axis=0)

nesting_summary = pd.DataFrame({
    "k9_cluster": ct_counts.index,
    "n_terms": ct_counts.sum(axis=1).values,
    "dominant_k2_cluster": ct_counts.idxmax(axis=1).values,
    "dominant_share_inside_k2": ct_share.max(axis=1).values,
})

nesting_overall = pd.DataFrame([{
    "ari_k2_k9": adjusted_rand_score(labels_k2, labels_k9),
    "nmi_k2_k9": normalized_mutual_info_score(labels_k2, labels_k9),
    "mean_k9_purity_inside_k2": nesting_summary["dominant_share_inside_k2"].mean(),
    "min_k9_purity_inside_k2": nesting_summary["dominant_share_inside_k2"].min(),
}])

ct_counts.to_csv(OUTPUT_DIR / "RB01_k2_vs_k9_crosstab_counts.csv")
ct_share.to_csv(OUTPUT_DIR / "RB01_k2_vs_k9_crosstab_row_shares.csv")
nesting_summary.to_csv(OUTPUT_DIR / "RB01_k2_vs_k9_nesting_summary.csv", index=False)
nesting_overall.to_csv(OUTPUT_DIR / "RB01_k2_vs_k9_nesting_overall.csv", index=False)

print("k=9 clusters nested inside k=2 macro-clusters:")
print(nesting_summary.to_string(index=False))
print("\nOverall nesting diagnostics:")
print(nesting_overall.to_string(index=False))

# Optional layered Ward hierarchy using existing LAYER_KS.
layered_labels, Z_layered = build_layered_clusters(features, layer_ks=LAYER_KS, method="ward")
layered_labels.to_csv(OUTPUT_DIR / "RB01_layered_ward_clusters.csv")

describe_layer_nesting(layered_labels).to_csv(
    OUTPUT_DIR / "RB01_layered_ward_nesting_summary.csv",
    index=False,
)


NameError: name 'consensus_matrices' is not defined

## 7.2 Replace/supplement delta-area with PAC

PAC is the fraction of consensus entries in an ambiguous interval. Lower PAC means less ambiguous clustering. This section also checks whether the delta-area choice changes under nearby thresholds.


In [ ]:
# ------------------------------------------------------------------
# PAC and delta-threshold sensitivity
# ------------------------------------------------------------------
def pac_score(C: np.ndarray, lower: float = 0.10, upper: float = 0.90) -> float:
    """Proportion of off-diagonal consensus entries in the ambiguous interval."""
    tri = C[np.triu_indices_from(C, k=1)]
    return float(((tri > lower) & (tri < upper)).mean())

pac_rows = []
for k in CONSENSUS_K_RANGE:
    Ck = consensus_matrices.get(k)
    if Ck is None:
        Ck = consensus_matrix(
            features.to_numpy(dtype=np.float32),
            k=k,
            n_resamples=CONSENSUS_N_RESAMPLES,
            subsample_frac=CONSENSUS_SUBSAMPLE_FRAC,
            random_state=RANDOM_STATE + k,
        )
        consensus_matrices[k] = Ck
    pac_rows.append({
        "k": k,
        "PAC_10_90": pac_score(Ck, 0.10, 0.90),
        "PAC_05_95": pac_score(Ck, 0.05, 0.95),
        "PAC_20_80": pac_score(Ck, 0.20, 0.80),
    })

pac_df = pd.DataFrame(pac_rows)
consensus_with_pac = consensus_scan.merge(pac_df, on="k", how="left")
consensus_with_pac["pac_rank_10_90"] = consensus_with_pac["PAC_10_90"].rank(method="min")

threshold_rows = []
for threshold in [0.015, 0.020, 0.030]:
    stable = consensus_scan[consensus_scan["delta_area"].abs() < threshold]
    selected_k_threshold = int(stable.iloc[0]["k"]) if not stable.empty else int(consensus_scan.loc[consensus_scan["consensus_cdf_area"].idxmax(), "k"])
    threshold_rows.append({
        "delta_threshold": threshold,
        "selected_k_by_first_small_delta": selected_k_threshold,
    })

delta_threshold_sensitivity = pd.DataFrame(threshold_rows)

consensus_with_pac.to_csv(OUTPUT_DIR / "RB02_consensus_scan_with_PAC.csv", index=False)
delta_threshold_sensitivity.to_csv(OUTPUT_DIR / "RB02_delta_threshold_sensitivity.csv", index=False)

print(consensus_with_pac.to_string(index=False))
print("\nDelta threshold sensitivity:")
print(delta_threshold_sensitivity.to_string(index=False))


## 7.3 SAX grid robustness beyond the single winning setting

This checks whether macro `k=2` and fine `k=9` partitions remain similar across several top SAX settings and different step fractions, rather than being an artifact of one window/overlap choice.


In [ ]:
# ------------------------------------------------------------------
# SAX robustness across top settings and alternative step fractions
# ------------------------------------------------------------------
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

SAX_ROBUST_TOP_N = 7
SAX_ROBUST_STEP_FRACTIONS = [0.10, 0.25, 0.50]
SAX_ROBUST_KS = [2, 9]

baseline_labels_by_k = {
    2: labels_k2,
    9: labels_k9,
}

base_top_settings = (
    ok_results
    .head(SAX_ROBUST_TOP_N)
    [["window_size", "n_segments", "alphabet_size"]]
    .drop_duplicates()
)

robust_rows = []
for _, setting in base_top_settings.iterrows():
    window_size = int(setting["window_size"])
    n_segments = int(setting["n_segments"])
    alphabet_size = int(setting["alphabet_size"])

    for step_frac in SAX_ROBUST_STEP_FRACTIONS:
        step = max(1, int(round(window_size * step_frac)))
        if not valid_sax_setting(window_size, step, n_segments, alphabet_size):
            continue

        try:
            f_alt = build_sax_feature_matrix_fast(
                array_dict=array_norm,
                window_size=window_size,
                n_segments=n_segments,
                alphabet_size=alphabet_size,
                step=step,
                spike_features=INCLUDE_SPIKE_FEATURES,
            )
            X_alt = f_alt.to_numpy(dtype=np.float32)

            for k in SAX_ROBUST_KS:
                C_alt = consensus_matrix(
                    X_alt,
                    k=k,
                    n_resamples=max(50, CONSENSUS_N_RESAMPLES // 2),
                    subsample_frac=CONSENSUS_SUBSAMPLE_FRAC,
                    random_state=RANDOM_STATE + 1000 + k + step,
                )
                lab_alt = consensus_labels(C_alt, k, f_alt.index)
                common = baseline_labels_by_k[k].index.intersection(lab_alt.index)
                robust_rows.append({
                    "window_size": window_size,
                    "n_segments": n_segments,
                    "alphabet_size": alphabet_size,
                    "step_fraction": step_frac,
                    "step": step,
                    "k": k,
                    "ari_vs_baseline": adjusted_rand_score(
                        baseline_labels_by_k[k].loc[common],
                        lab_alt.loc[common],
                    ),
                    "nmi_vs_baseline": normalized_mutual_info_score(
                        baseline_labels_by_k[k].loc[common],
                        lab_alt.loc[common],
                    ),
                    "pac_10_90": pac_score(C_alt, 0.10, 0.90),
                })
        except Exception as e:
            robust_rows.append({
                "window_size": window_size,
                "n_segments": n_segments,
                "alphabet_size": alphabet_size,
                "step_fraction": step_frac,
                "step": step,
                "k": np.nan,
                "status": "failed",
                "error": str(e),
            })

sax_robustness = pd.DataFrame(robust_rows)
sax_robustness.to_csv(OUTPUT_DIR / "RB03_sax_grid_step_robustness.csv", index=False)
print(sax_robustness.sort_values(["k", "ari_vs_baseline"], ascending=[True, False]).to_string(index=False))


## 7.4 Preprocessing constant sensitivity

Vary detrending and denoising windows independently. The main red-flag check is whether cluster count, cluster membership, and the external predictive result survive reasonable nearby constants.


In [ ]:
# ------------------------------------------------------------------
# Preprocessing ablation: DETREND_WINDOW and DENOISE_WINDOW
# ------------------------------------------------------------------
def preprocess_panel_configurable(
    panel: pd.DataFrame,
    denoise_window: int = DENOISE_WINDOW,
    detrend_window: int = DETREND_WINDOW,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    filled = panel.apply(interpolate_small_gaps, axis=0)
    denoised = filled.apply(lambda s: denoise_series(s, window=denoise_window), axis=0)
    detrended = denoised.apply(lambda s: detrend_series(s, window=detrend_window), axis=0)
    normalized = detrended.apply(robust_zscore_series, axis=0)
    soft = soft_cap_panel(normalized, cap=10.0)
    return normalized, {
        "filled": filled,
        "denoised": denoised,
        "detrended": detrended,
        "normalized": normalized,
        "soft": soft,
    }

DENOISE_ABLATION_WINDOWS = [9, 15, 21]
DETREND_ABLATION_WINDOWS = [60, 91, 120]
BASE_DENOISE_WINDOW = DENOISE_WINDOW
BASE_DETREND_WINDOW = DETREND_WINDOW

ablation_specs = []
for w in DENOISE_ABLATION_WINDOWS:
    ablation_specs.append({"ablation_type": "denoise", "denoise_window": w, "detrend_window": BASE_DETREND_WINDOW})
for w in DETREND_ABLATION_WINDOWS:
    ablation_specs.append({"ablation_type": "detrend", "denoise_window": BASE_DENOISE_WINDOW, "detrend_window": w})

# remove duplicate baseline if included twice
ablation_specs = [dict(t) for t in {tuple(sorted(d.items())) for d in ablation_specs}]

ablation_rows = []
for spec in ablation_specs:
    try:
        p_alt, _ = preprocess_panel_configurable(
            panel_raw_for_clustering,
            denoise_window=spec["denoise_window"],
            detrend_window=spec["detrend_window"],
        )
        arr_alt = panel_to_array_dict(p_alt)
        f_alt = build_sax_feature_matrix_fast(
            array_dict=arr_alt,
            window_size=selected_params["window_size"],
            n_segments=selected_params["n_segments"],
            alphabet_size=selected_params["alphabet_size"],
            step=selected_params["step"],
            spike_features=selected_params["spike_features"],
        )
        eval_alt = evaluate_clustering_for_features(f_alt, k_grid=K_GRID, silhouette_sample_size=None)
        best_alt = eval_alt[eval_alt["status"] == "ok"].iloc[0]

        labels_alt_k2, _ = run_kmeans_final(f_alt, k=2)
        labels_alt_k9, _ = run_kmeans_final(f_alt, k=9)

        common2 = labels_k2.index.intersection(labels_alt_k2.index)
        common9 = labels_k9.index.intersection(labels_alt_k9.index)

        row = {
            **spec,
            "selected_k_by_silhouette": int(best_alt["k"]),
            "best_silhouette": float(best_alt["silhouette"]),
            "ari_k2_vs_baseline": adjusted_rand_score(labels_k2.loc[common2], labels_alt_k2.loc[common2]),
            "ari_k9_vs_baseline": adjusted_rand_score(labels_k9.loc[common9], labels_alt_k9.loc[common9]),
        }

        # Optional: check whether cluster-2 external correlation survives if market analysis objects exist.
        if "market_panel" in globals():
            cluster_idx_alt = build_cluster_indices(p_alt, labels_alt_k2)
            target = market_panel[target_col].reindex(cluster_idx_alt.index)
            corr_alt = cluster_idx_alt.corrwith(target, axis=0)
            row["max_abs_cluster_corr_with_target"] = float(corr_alt.abs().max())
            row["best_corr_cluster"] = corr_alt.abs().idxmax()

        ablation_rows.append(row)
    except Exception as e:
        ablation_rows.append({**spec, "status": "failed", "error": str(e)})

preprocessing_ablation = pd.DataFrame(ablation_rows)
preprocessing_ablation.to_csv(OUTPUT_DIR / "RB04_preprocessing_constant_sensitivity.csv", index=False)
print(preprocessing_ablation.to_string(index=False))


## 7.5 Stable-core and medoid drill-down

For each cluster, identify the medoid term and the top stable members. Use this stable core for cleaner plots and a stricter version of the external predictive test.


In [ ]:
# ------------------------------------------------------------------
# Stable core: medoid + top stable terms per cluster
# ------------------------------------------------------------------
from sklearn.metrics import pairwise_distances

STABLE_CORE_TOP_K = 20
STABLE_CORE_MIN_K = 5

X_scaled = StandardScaler().fit_transform(features.to_numpy(dtype=np.float32))
X_scaled_df = pd.DataFrame(X_scaled, index=features.index)

stable_core_rows = []
for cluster_id, grp in consensus_table.groupby("cluster"):
    terms = grp["term"].tolist()
    terms = [t for t in terms if t in X_scaled_df.index]
    if not terms:
        continue

    D = pairwise_distances(X_scaled_df.loc[terms], metric="euclidean")
    medoid_term = terms[int(D.sum(axis=1).argmin())]

    top_stable_terms = (
        grp.sort_values("stability_score", ascending=False)
        .head(max(STABLE_CORE_MIN_K, min(STABLE_CORE_TOP_K, len(grp))))["term"]
        .tolist()
    )

    for term in top_stable_terms:
        stable_core_rows.append({
            "cluster": cluster_id,
            "term": term,
            "is_medoid": term == medoid_term,
            "stability_score": float(grp.loc[grp["term"] == term, "stability_score"].iloc[0]),
        })
    if medoid_term not in top_stable_terms:
        stable_core_rows.append({
            "cluster": cluster_id,
            "term": medoid_term,
            "is_medoid": True,
            "stability_score": float(grp.loc[grp["term"] == medoid_term, "stability_score"].iloc[0]),
        })

stable_core_table = pd.DataFrame(stable_core_rows).sort_values(
    ["cluster", "is_medoid", "stability_score"],
    ascending=[True, False, False],
)
stable_core_table.to_csv(OUTPUT_DIR / "RB05_stable_core_terms_and_medoids.csv", index=False)

# Rebuild cluster indices using stable-core terms only.
stable_core_labels = stable_core_table.set_index("term")["cluster"]
stable_panel = panel_norm[panel_norm.columns.intersection(stable_core_labels.index)]
stable_cluster_indices = build_cluster_indices(stable_panel, stable_core_labels)
stable_cluster_indices.to_csv(OUTPUT_DIR / "RB05_stable_core_cluster_indices.csv")

if "market_panel" in globals():
    stable_analysis_df = stable_cluster_indices.join(market_panel[[target_col]], how="inner").dropna()
    stable_corr = stable_analysis_df.drop(columns=[target_col]).corrwith(stable_analysis_df[target_col])
    stable_corr.rename("corr_with_sp500_vol").to_csv(OUTPUT_DIR / "RB05_stable_core_sp500_corr.csv")
    print(stable_corr.sort_values(ascending=False).to_string())

print(stable_core_table.head(60).to_string(index=False))


## 7.6 Event-concentration audit

This collapses consecutive high-attention days into distinct event episodes. It helps distinguish broad recurring predictive power from a result driven by one or two long-duration episodes.


In [ ]:
# ------------------------------------------------------------------
# Event concentration audit for stable-core terms
# ------------------------------------------------------------------
def extract_event_episodes(
    s: pd.Series,
    threshold: float | None = None,
    threshold_quantile: float = 0.95,
    min_gap_days: int = 7,
) -> pd.DataFrame:
    """Collapse consecutive above-threshold days into episodes for one term."""
    x = s.dropna().astype(float)
    if x.empty:
        return pd.DataFrame(columns=["start", "end", "duration_days", "peak_date", "peak_value"])
    if threshold is None:
        threshold = x.quantile(threshold_quantile)
    hit_dates = x.index[x >= threshold]
    if len(hit_dates) == 0:
        return pd.DataFrame(columns=["start", "end", "duration_days", "peak_date", "peak_value"])

    episodes = []
    start = hit_dates[0]
    prev = hit_dates[0]
    for dt in hit_dates[1:]:
        if (dt - prev).days <= min_gap_days:
            prev = dt
        else:
            seg = x.loc[start:prev]
            episodes.append({
                "start": start,
                "end": prev,
                "duration_days": (prev - start).days + 1,
                "peak_date": seg.idxmax(),
                "peak_value": float(seg.max()),
            })
            start = prev = dt
    seg = x.loc[start:prev]
    episodes.append({
        "start": start,
        "end": prev,
        "duration_days": (prev - start).days + 1,
        "peak_date": seg.idxmax(),
        "peak_value": float(seg.max()),
    })
    return pd.DataFrame(episodes)

EVENT_THRESHOLD_Q = 0.95
EVENT_MIN_GAP_DAYS = 7

episode_rows = []
for _, row in stable_core_table.iterrows():
    term = row["term"]
    cluster_id = row["cluster"]
    if term not in panel_norm.columns:
        continue
    episodes = extract_event_episodes(
        panel_norm[term],
        threshold_quantile=EVENT_THRESHOLD_Q,
        min_gap_days=EVENT_MIN_GAP_DAYS,
    )
    for _, ep in episodes.iterrows():
        episode_rows.append({
            "cluster": cluster_id,
            "term": term,
            "start": ep["start"],
            "end": ep["end"],
            "duration_days": ep["duration_days"],
            "peak_date": ep["peak_date"],
            "peak_value": ep["peak_value"],
        })

event_episodes = pd.DataFrame(episode_rows)
event_episodes.to_csv(OUTPUT_DIR / "RB06_stable_core_event_episodes.csv", index=False)

if not event_episodes.empty:
    event_concentration = (
        event_episodes
        .groupby("cluster")
        .agg(
            n_terms=("term", "nunique"),
            n_episodes=("peak_date", "count"),
            median_duration_days=("duration_days", "median"),
            max_duration_days=("duration_days", "max"),
        )
        .reset_index()
    )
    top_event_share = []
    for c, g in event_episodes.groupby("cluster"):
        peak_month = g["peak_date"].dt.to_period("M").astype(str)
        top_share = peak_month.value_counts(normalize=True).iloc[0]
        top_event_share.append({"cluster": c, "largest_peak_month_share": float(top_share)})
    event_concentration = event_concentration.merge(pd.DataFrame(top_event_share), on="cluster", how="left")
    event_concentration.to_csv(OUTPUT_DIR / "RB06_event_concentration_summary.csv", index=False)
    print(event_concentration.to_string(index=False))
else:
    print("No stable-core event episodes detected at the chosen threshold.")


## 7.7 Permutation/placebo test for cluster predictive power

Randomly draw same-size placebo clusters from the available term universe and compare their rolling/predictive correlation with the observed target cluster. The empirical p-value is the share of placebo draws that equal or beat the observed statistic.


In [ ]:
# ------------------------------------------------------------------
# Placebo cluster test for external predictiveness
# ------------------------------------------------------------------
PLACEBO_N_DRAWS = 1000
PLACEBO_TARGET_CLUSTER = 2
PLACEBO_RANDOM_STATE = 42
PLACEBO_USE_STABLE_CORE = False

rng = np.random.default_rng(PLACEBO_RANDOM_STATE)

if "market_panel" not in globals():
    raise RuntimeError("Run the external S&P 500 validation section before the placebo test.")

label_source = stable_core_labels if PLACEBO_USE_STABLE_CORE else labels_consensus
panel_source = stable_panel if PLACEBO_USE_STABLE_CORE else panel_norm

cluster_terms = label_source[label_source == PLACEBO_TARGET_CLUSTER].index.intersection(panel_source.columns).tolist()
all_terms = panel_source.columns.tolist()
cluster_size = len(cluster_terms)

if cluster_size == 0:
    raise ValueError(f"No terms found for cluster {PLACEBO_TARGET_CLUSTER}.")

observed_index = panel_source[cluster_terms].mean(axis=1).rename("observed_cluster_index")
observed_df = pd.concat([observed_index, market_panel[target_col]], axis=1).dropna()
observed_corr = float(observed_df["observed_cluster_index"].corr(observed_df[target_col]))
observed_abs_corr = abs(observed_corr)

placebo_rows = []
for draw in range(PLACEBO_N_DRAWS):
    sampled_terms = rng.choice(all_terms, size=cluster_size, replace=False)
    placebo_index = panel_source[list(sampled_terms)].mean(axis=1)
    placebo_df = pd.concat([placebo_index.rename("placebo"), market_panel[target_col]], axis=1).dropna()
    corr = float(placebo_df["placebo"].corr(placebo_df[target_col]))
    placebo_rows.append({
        "draw": draw,
        "corr": corr,
        "abs_corr": abs(corr),
    })

placebo_df = pd.DataFrame(placebo_rows)
empirical_p_two_sided = float((placebo_df["abs_corr"] >= observed_abs_corr).mean())
empirical_p_positive = float((placebo_df["corr"] >= observed_corr).mean())

placebo_summary = pd.DataFrame([{
    "target_cluster": PLACEBO_TARGET_CLUSTER,
    "cluster_size": cluster_size,
    "observed_corr": observed_corr,
    "observed_abs_corr": observed_abs_corr,
    "empirical_p_two_sided_abs_corr": empirical_p_two_sided,
    "empirical_p_positive_corr": empirical_p_positive,
    "n_placebo_draws": PLACEBO_N_DRAWS,
    "used_stable_core": PLACEBO_USE_STABLE_CORE,
}])

placebo_df.to_csv(OUTPUT_DIR / "RB07_placebo_cluster_draws.csv", index=False)
placebo_summary.to_csv(OUTPUT_DIR / "RB07_placebo_cluster_summary.csv", index=False)

print(placebo_summary.to_string(index=False))


## 8. Clinic brief reporting checklist

Use the robustness tables as decision rules:

- If `RB01_k2_vs_k9_nesting_summary.csv` shows high fine-cluster purity inside macro clusters, present the result as a hierarchy rather than choosing between `k=2` and `k=9`.
- If PAC and delta-threshold sensitivity disagree, report `k=9` as exploratory rather than definitive.
- If SAX and preprocessing ablations preserve the macro split and external validation, emphasize the shape result as robust.
- If stable-core and event-concentration audits show the predictive result survives many independent episodes, the cluster-2 story is much stronger.
- If placebo p-values are small, the result is less likely to be a data-mined artifact.


# Poster tables

In [ ]:
# -----------------------------------------------------------------------------
# External market benchmark + cluster index analysis
# Do NOT add market indices to the clustering panel.
# Use them only after clustering as external outcomes/benchmarks.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error


# -----------------------------------------------------------------------------
# Paths
# -----------------------------------------------------------------------------

OUTPUT_DIR = Path(r"C:\Python\CSUREMM\output\sax_tests_july_08")

CONSENSUS_PATH = OUTPUT_DIR / "11_consensus_labels.csv"

SP500_PATH = Path(r"C:\Python\CSUREMM\shock_detection\SP500_data.csv")
DJ_PATH = Path(r"C:\Python\CSUREMM\shock_detection\DOWJONES_data.csv")
NASDAQ_PATH = Path(r"C:\Python\CSUREMM\shock_detection\NASDAQ100_data.csv")
RUSSELL_PATH = Path(r"C:\Python\CSUREMM\shock_detection\RUSSELL2000_data.csv")

MARKET_PATHS = {
    "sp500": SP500_PATH,
    "dowjones": DJ_PATH,
    "nasdaq100": NASDAQ_PATH,
    "russell2000": RUSSELL_PATH,
}

PRICE_COL = "price"
VOL_WINDOW = 7

HORIZONS = [1, 7, 14,]
INITIAL_TRAIN_FRAC = 0.70
MIN_TRAIN_N = 250
MIN_TEST_N = 60

MARKET_DIR = OUTPUT_DIR / "consensus_cluster_market_analysis"
MARKET_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# Load consensus labels
# -----------------------------------------------------------------------------

consensus_table = pd.read_csv(CONSENSUS_PATH)

required_cols = {"term", "cluster", "stability_score"}
missing = required_cols - set(consensus_table.columns)

if missing:
    raise ValueError(f"Consensus file missing columns: {missing}")

consensus_table["term"] = consensus_table["term"].astype(str)
consensus_table["cluster"] = consensus_table["cluster"].astype(int)
consensus_table["stability_score"] = consensus_table["stability_score"].astype(float)


# -----------------------------------------------------------------------------
# Load market price, return, and realized volatility
# -----------------------------------------------------------------------------

def load_market_panel(
    path: Path,
    market_name: str,
    price_col: str = PRICE_COL,
    vol_window: int = VOL_WINDOW,
) -> pd.DataFrame:

    df = (
        pd.read_csv(path, parse_dates=["Time"])
        .dropna(subset=["Time"])
        .drop_duplicates("Time")
        .sort_values("Time")
        .set_index("Time")
    )

    if price_col not in df.columns:
        raise ValueError(
            f"{market_name}: column `{price_col}` not found. "
            f"Available columns: {list(df.columns)}"
        )

    price = df[price_col].astype(float)
    log_return = np.log(price).diff()

    realized_vol = (
    np.sqrt(
        log_return.pow(2)
        .rolling(vol_window)
        .sum()
    )
    )

    # originally this:
    # realized_vol = ( log_return .rolling( window=vol_window, min_periods=max(3, vol_window // 2), ) .std() )

    out = pd.DataFrame({
        f"{market_name}_price": price,
        f"{market_name}_log_return": log_return,
        f"{market_name}_realized_volatility_{vol_window}d": realized_vol,
    })

    return out


market_panels = {
    name: load_market_panel(path, name)
    for name, path in MARKET_PATHS.items()
}

market_all = pd.concat(market_panels.values(), axis=1)
market_all.to_csv(MARKET_DIR / "all_market_panels.csv")

In [38]:
# # -----------------------------------------------------------------------------
# # Construct one Google Trends index per consensus cluster
# # Top 25% most stable terms only, stability-weighted
# # -----------------------------------------------------------------------------

def build_top_quartile_cluster_indices(
    panel_norm: pd.DataFrame,
    consensus_table: pd.DataFrame,
    top_frac: float = 0.25,
) -> tuple[pd.DataFrame, pd.DataFrame]:

    index_dict = {}
    member_rows = []

    for cluster_id in sorted(consensus_table["cluster"].unique()):

        cluster_terms_df = (
            consensus_table
            .loc[consensus_table["cluster"] == cluster_id]
            .copy()
        )

        cluster_terms_df = cluster_terms_df[
            cluster_terms_df["term"].isin(panel_norm.columns)
        ]

        if len(cluster_terms_df) == 0:
            continue

        n_keep = max(1, int(np.ceil(len(cluster_terms_df) * top_frac)))

        cluster_terms_df = (
            cluster_terms_df
            .sort_values("stability_score", ascending=False)
            .head(n_keep)
            .copy()
        )

        terms = cluster_terms_df["term"].tolist()
        X = panel_norm[terms].copy()

        w = cluster_terms_df["stability_score"].astype(float).copy()
        w = w - w.min()

        if w.sum() == 0:
            w = pd.Series(np.ones(len(w)), index=w.index)

        w = w / w.sum()

        index_name = f"cluster_{cluster_id}_index"

        index_series = X.mul(w.to_numpy(), axis=1).sum(axis=1)
        index_series.name = index_name

        index_dict[index_name] = index_series

        temp_members = cluster_terms_df.copy()
        temp_members["cluster_index"] = index_name
        temp_members["weight"] = w.to_numpy()
        temp_members["n_cluster_terms_available"] = len(
            consensus_table.loc[consensus_table["cluster"] == cluster_id]
        )
        temp_members["n_terms_used_top25pct"] = n_keep

        member_rows.append(temp_members)

    indices = pd.DataFrame(index_dict)
    members = pd.concat(member_rows, ignore_index=True)

    return indices, members


cluster_indices, cluster_index_members = build_top_quartile_cluster_indices(
    panel_norm=panel_norm,
    consensus_table=consensus_table,
    top_frac=0.25,
)

cluster_indices.to_csv(MARKET_DIR / "cluster_indices_top25pct_stability_weighted.csv")
cluster_index_members.to_csv(
    MARKET_DIR / "cluster_index_members_top25pct_stability_weighted.csv",
    index=False,
)

# -----------------------------------------------------------------------------
# Construct one Google Trends index per consensus cluster
# Uses all cluster terms, stability-score weighted
# -----------------------------------------------------------------------------

# def build_stability_weighted_cluster_indices(
#     panel_norm: pd.DataFrame,
#     consensus_table: pd.DataFrame,
# ) -> tuple[pd.DataFrame, pd.DataFrame]:

#     index_dict = {}
#     member_rows = []

#     for cluster_id in sorted(consensus_table["cluster"].unique()):

#         cluster_terms_df = (
#             consensus_table
#             .loc[consensus_table["cluster"] == cluster_id]
#             .copy()
#         )

#         cluster_terms_df = cluster_terms_df[
#             cluster_terms_df["term"].isin(panel_norm.columns)
#         ]

#         if len(cluster_terms_df) == 0:
#             continue

#         terms = cluster_terms_df["term"].tolist()
#         X = panel_norm[terms].copy()

#         w = cluster_terms_df["stability_score"].astype(float).copy()

#         # Shift weights to be nonnegative within cluster
#         w = w - w.min()

#         if w.sum() == 0:
#             w = pd.Series(np.ones(len(w)), index=w.index)

#         w = w / w.sum()

#         index_name = f"cluster_{cluster_id}_index"

#         index_series = X.mul(w.to_numpy(), axis=1).sum(axis=1)
#         index_series.name = index_name

#         index_dict[index_name] = index_series

#         temp_members = cluster_terms_df.copy()
#         temp_members["cluster_index"] = index_name
#         temp_members["weight"] = w.to_numpy()
#         temp_members["n_terms_used"] = len(cluster_terms_df)

#         member_rows.append(temp_members)

#     indices = pd.DataFrame(index_dict)
#     members = pd.concat(member_rows, ignore_index=True)

#     return indices, members


# cluster_indices, cluster_index_members = build_stability_weighted_cluster_indices(
#     panel_norm=panel_norm,
#     consensus_table=consensus_table,
# )

# cluster_indices.to_csv(MARKET_DIR / "cluster_indices_all_terms_stability_weighted.csv")
# cluster_index_members.to_csv(
#     MARKET_DIR / "cluster_index_members_all_terms_stability_weighted.csv",
#     index=False,
# )

# -----------------------------------------------------------------------------
# Utility metrics
# -----------------------------------------------------------------------------

def oos_r2(y_true: np.ndarray, y_pred: np.ndarray, y_base: np.ndarray) -> float:
    sse_model = np.sum((y_true - y_pred) ** 2)
    sse_base = np.sum((y_true - y_base) ** 2)

    if sse_base == 0:
        return np.nan

    return 1 - sse_model / sse_base


def directional_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    true_direction = np.sign(np.diff(y_true))
    pred_direction = np.sign(np.diff(y_pred))

    valid = true_direction != 0

    if valid.sum() == 0:
        return np.nan

    return np.mean(true_direction[valid] == pred_direction[valid])


def oos_corr(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    if len(y_true) < 3:
        return np.nan

    return pd.Series(y_true).corr(pd.Series(y_pred))


def assign_stability_flag(
    oos_r2_value: float,
    delta_vs_baseline: float,
    directional_acc: float,
    test_n: int,
) -> str:

    if test_n < MIN_TEST_N:
        return "provisional_low_n"

    if (
        pd.notna(oos_r2_value)
        and pd.notna(delta_vs_baseline)
        and pd.notna(directional_acc)
        and oos_r2_value > 0
        and delta_vs_baseline > 0
        and directional_acc >= 0.52
    ):
        return "stable"

    if (
        pd.notna(delta_vs_baseline)
        and pd.notna(directional_acc)
        and delta_vs_baseline > 0
        and directional_acc >= 0.50
    ):
        return "provisional"

    return "weak"


# -----------------------------------------------------------------------------
# OOS predictive test:
# target_{t+h} = alpha + beta * cluster_index_t
#
# Baseline:
# target_{t+h} predicted by current realized volatility target_t
# -----------------------------------------------------------------------------

def run_oos_predictive_tests(
    df: pd.DataFrame,
    index_cols: list[str],
    target_col: str,
    market_name: str,
    horizons: list[int],
    initial_train_frac: float = INITIAL_TRAIN_FRAC,
) -> tuple[pd.DataFrame, pd.DataFrame]:

    poster_rows = []
    appendix_rows = []

    for cluster_col in index_cols:

        for h in horizons:

            reg_df = pd.DataFrame({
                "x_cluster": df[cluster_col],
                "x_baseline": df[target_col],
                "target": df[target_col].shift(-h),
            }).dropna()

            if len(reg_df) < MIN_TRAIN_N + MIN_TEST_N:
                continue

            initial_train_n = max(
                MIN_TRAIN_N,
                int(len(reg_df) * initial_train_frac),
            )

            if len(reg_df) - initial_train_n < MIN_TEST_N:
                continue

            y_true_all = []
            y_pred_model_all = []
            y_pred_base_all = []

            for t in range(initial_train_n, len(reg_df)):

                train = reg_df.iloc[:t]
                test = reg_df.iloc[[t]]

                model = LinearRegression()
                model.fit(train[["x_cluster"]], train["target"])

                base_model = LinearRegression()
                base_model.fit(train[["x_baseline"]], train["target"])

                y_true_all.append(float(test["target"].iloc[0]))
                y_pred_model_all.append(
                    float(model.predict(test[["x_cluster"]])[0])
                )
                y_pred_base_all.append(
                    float(base_model.predict(test[["x_baseline"]])[0])
                )

            y_true = np.array(y_true_all)
            y_pred_model = np.array(y_pred_model_all)
            y_pred_base = np.array(y_pred_base_all)

            model_oos_r2 = oos_r2(y_true, y_pred_model, y_pred_base)
            base_rmse = mean_squared_error(y_true, y_pred_base) ** 0.5
            model_rmse = mean_squared_error(y_true, y_pred_model) ** 0.5

            model_oos_corr = oos_corr(y_true, y_pred_model)
            base_oos_corr = oos_corr(y_true, y_pred_base)

            dir_acc = directional_accuracy(y_true, y_pred_model)
            base_dir_acc = directional_accuracy(y_true, y_pred_base)

            delta_vs_baseline = model_oos_r2
            rmse_ratio_vs_baseline = model_rmse / base_rmse if base_rmse != 0 else np.nan
            dir_acc_delta_vs_baseline = dir_acc - base_dir_acc

            stability_flag = assign_stability_flag(
                oos_r2_value=model_oos_r2,
                delta_vs_baseline=delta_vs_baseline,
                directional_acc=dir_acc,
                test_n=len(y_true),
            )

            poster_rows.append({
                "market_index": market_name,
                "cluster_index": cluster_col,
                "target": target_col,
                "horizon_days": h,

                "poster_oos_r2_vs_lagged_vol_baseline": model_oos_r2,
                "poster_oos_corr": model_oos_corr,
                "poster_directional_accuracy": dir_acc,
                "poster_delta_vs_baseline": delta_vs_baseline,
                "poster_stability_flag": stability_flag,
            })

            appendix_rows.append({
                "market_index": market_name,
                "cluster_index": cluster_col,
                "target": target_col,
                "horizon_days": h,

                "initial_train_n": initial_train_n,
                "test_n": len(y_true),

                "model_oos_r2_vs_baseline": model_oos_r2,
                "model_oos_corr": model_oos_corr,
                "baseline_oos_corr": base_oos_corr,

                "model_directional_accuracy": dir_acc,
                "baseline_directional_accuracy": base_dir_acc,
                "directional_accuracy_delta_vs_baseline": dir_acc_delta_vs_baseline,

                "model_oos_rmse": model_rmse,
                "baseline_oos_rmse": base_rmse,
                "rmse_ratio_vs_baseline": rmse_ratio_vs_baseline,

                "stability_flag": stability_flag,
            })

    return pd.DataFrame(poster_rows), pd.DataFrame(appendix_rows)


# -----------------------------------------------------------------------------
# Run tests for each market index
# -----------------------------------------------------------------------------

all_poster_results = []
all_appendix_results = []

for market_name in MARKET_PATHS:

    target_col = f"{market_name}_realized_volatility_{VOL_WINDOW}d"

    analysis_df = (
        cluster_indices
        .join(market_panels[market_name], how="inner")
        .dropna()
    )

    analysis_df.to_csv(
        MARKET_DIR / f"cluster_indices_with_{market_name}.csv"
    )

    poster_df, appendix_df = run_oos_predictive_tests(
        df=analysis_df,
        index_cols=list(cluster_indices.columns),
        target_col=target_col,
        market_name=market_name,
        horizons=HORIZONS,
    )

    all_poster_results.append(poster_df)
    all_appendix_results.append(appendix_df)


poster_results = pd.concat(all_poster_results, ignore_index=True)
appendix_results = pd.concat(all_appendix_results, ignore_index=True)

poster_results.to_csv(
    MARKET_DIR / "POSTER_core_four_metrics_cluster_by_market.csv",
    index=False,
)

appendix_results.to_csv(
    MARKET_DIR / "APPENDIX_full_oos_metrics_cluster_by_market.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Lead-lag correlation appendix
# Corr(cluster_index_t, market_target_{t+h})
# h > 0 means cluster index leads market volatility.
# -----------------------------------------------------------------------------

def lead_lag_correlation(
    df: pd.DataFrame,
    index_cols: list[str],
    target_col: str,
    market_name: str,
    max_lag: int = 30,
) -> pd.DataFrame:

    rows = []

    for index_col in index_cols:
        for h in range(-max_lag, max_lag + 1):

            corr = df[index_col].corr(df[target_col].shift(-h))

            rows.append({
                "market_index": market_name,
                "cluster_index": index_col,
                "target": target_col,
                "horizon_days": h,
                "correlation": corr,
            })

    return pd.DataFrame(rows)


lead_lag_all = []

for market_name in MARKET_PATHS:

    target_col = f"{market_name}_realized_volatility_{VOL_WINDOW}d"

    analysis_df = (
        cluster_indices
        .join(market_panels[market_name], how="inner")
        .dropna()
    )

    temp_corr = lead_lag_correlation(
        df=analysis_df,
        index_cols=list(cluster_indices.columns),
        target_col=target_col,
        market_name=market_name,
        max_lag=30,
    )

    lead_lag_all.append(temp_corr)

lead_lag_corr = pd.concat(lead_lag_all, ignore_index=True)

lead_lag_corr.to_csv(
    MARKET_DIR / "APPENDIX_lead_lag_correlations_cluster_by_market.csv",
    index=False,
)


# -----------------------------------------------------------------------------
# Poster table: best horizon per market × cluster
# Prefer OOS R² when positive; otherwise OOS corr is kept as backup.
# -----------------------------------------------------------------------------

poster_best = (
    poster_results
    .sort_values(
        [
            "market_index",
            "cluster_index",
            "poster_oos_r2_vs_lagged_vol_baseline",
            "poster_oos_corr",
        ],
        ascending=[True, True, False, False],
    )
    .groupby(["market_index", "cluster_index"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

poster_best.to_csv(
    MARKET_DIR / "POSTER_best_horizon_core_four_metrics.csv",
    index=False,
)

print("\nPoster-ready best horizon table:")
print(
    poster_best
    .sort_values(
        ["market_index", "poster_oos_r2_vs_lagged_vol_baseline"],
        ascending=[True, False],
    )
    .to_string(index=False)
)


# -----------------------------------------------------------------------------
# Compact heatmap-style CSV pivot for easier poster formatting
# -----------------------------------------------------------------------------

poster_pivot = poster_best.pivot_table(
    index="cluster_index",
    columns="market_index",
    values="poster_oos_r2_vs_lagged_vol_baseline",
    aggfunc="first",
)

poster_pivot.to_csv(
    MARKET_DIR / "POSTER_oos_r2_heatmap_matrix.csv"
)


# -----------------------------------------------------------------------------
# Optional plots: OOS R² by market and cluster
# -----------------------------------------------------------------------------

for market_name in MARKET_PATHS:

    temp = poster_best[poster_best["market_index"] == market_name].copy()
    temp = temp.sort_values("poster_oos_r2_vs_lagged_vol_baseline", ascending=False)

    plt.figure(figsize=(10, 5))

    plt.bar(
        temp["cluster_index"],
        temp["poster_oos_r2_vs_lagged_vol_baseline"],
    )

    plt.axhline(0, linewidth=1, alpha=0.6)

    plt.title(f"Best-Horizon OOS R² vs Lagged-Vol Baseline: {market_name}")
    plt.xlabel("Cluster index")
    plt.ylabel("OOS R² vs baseline")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    plt.savefig(
        MARKET_DIR / f"POSTER_oos_r2_by_cluster_{market_name}.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()


Poster-ready best horizon table:
market_index   cluster_index                             target  horizon_days  poster_oos_r2_vs_lagged_vol_baseline  poster_oos_corr  poster_directional_accuracy  poster_delta_vs_baseline poster_stability_flag
    dowjones cluster_0_index    dowjones_realized_volatility_7d            14                              0.086337         0.430853                     0.492308                  0.086337                  weak
    dowjones cluster_2_index    dowjones_realized_volatility_7d            14                              0.035290         0.172843                     0.556923                  0.035290                stable
    dowjones cluster_4_index    dowjones_realized_volatility_7d            14                              0.020461         0.066002                     0.532308                  0.020461                stable
    dowjones cluster_3_index    dowjones_realized_volatility_7d            14                              0.008513         0.

# Poster figure on SAX rendering

In [24]:
# -------------------------------------------------------------------------
# Minimal SAX helpers (poster figure only)
# -------------------------------------------------------------------------

ALPHABET_SIZE = 5
from scipy.stats import norm

# Global Gaussian breakpoints used by SAX
BREAKPOINTS = norm.ppf(
    np.linspace(0, 1, ALPHABET_SIZE + 1)[1:-1]
)


def paa(x: np.ndarray, n_segments: int = 16) -> np.ndarray:
    """
    Piecewise Aggregate Approximation (PAA).
    Works even when len(x) is not divisible by n_segments.
    """
    x = np.asarray(x, dtype=float)

    edges = np.linspace(0, len(x), n_segments + 1)

    out = np.empty(n_segments)

    for i in range(n_segments):
        lo = int(np.floor(edges[i]))
        hi = int(np.floor(edges[i + 1]))

        if hi <= lo:
            hi = lo + 1

        out[i] = np.mean(x[lo:hi])

    return out


def value_to_symbol(v: float, breakpoints=BREAKPOINTS) -> str:
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    return alphabet[np.searchsorted(breakpoints, v)]


def sax_word(window: np.ndarray) -> str:
    z = (window - window.mean()) / window.std(ddof=0)
    seg = paa(z)
    return "".join(value_to_symbol(v) for v in seg)

In [29]:
# -------------------------------------------------------------------------
# SAX explainer figure for poster
# -------------------------------------------------------------------------

SAX_EXAMPLE_TERM = "tesla_stock_price"   # use clean term name, not folder name

def choose_sax_example_term(panel: pd.DataFrame, preferred: str | None = None) -> str:
    if preferred in panel.columns:
        return preferred

    # Fallback: choose a well-observed, visibly varying term.
    score = (
        panel
        .notna()
        .sum()
        .rank(pct=True)
        + panel
        .std(skipna=True)
        .rank(pct=True)
    )

    return score.sort_values(ascending=False).index[0]


def plot_sax_example(
    panel: pd.DataFrame,
    term: str,
    path: Path,
    window_size: int = 60,
    n_segments: int = 16,
):
    s = panel[term].dropna()

    # Use the first high-variance local window, not necessarily the first window.
    best_start = 0
    best_sd = -np.inf

    for start in range(0, max(1, len(s) - window_size), max(1, window_size // 4)):
        w = s.iloc[start:start + window_size]

        if len(w) == window_size and w.std() > best_sd:
            best_sd = w.std()
            best_start = start

    w = s.iloc[best_start:best_start + window_size].copy()

    z = (w - w.mean()) / w.std(ddof=0)
    z = z.clip(-3, 6)

    seg = paa(z.values)
    word = sax_word(w.values)

    x = np.arange(len(z))

    fig, ax = plt.subplots(figsize=(8.6, 4.4))

    # Background raw z-normalized curve.
    ax.plot(
        x,
        z.values,
        color="0.72",
        linewidth=1.8,
        alpha=0.75,
        label="Z-normalized series",
    )

    # Segment boundaries and PAA levels.
    for i, val in enumerate(seg):
        lo = int(i * window_size / n_segments)
        hi = int((i + 1) * window_size / n_segments)

        ax.axvline(
            lo,
            color="0.85",
            linewidth=0.8,
            linestyle="--",
            alpha=0.8,
        )

        ax.hlines(
            val,
            lo,
            hi,
            color="red",
            linewidth=2.6,
        )

        ax.text(
            (lo + hi) / 2,
            val + 0.18,
            word[i],
            ha="center",
            va="bottom",
            fontsize=11,
            fontweight="bold",
            color="0.20",
        )

    ax.axvline(
        window_size,
        color="0.85",
        linewidth=0.8,
        linestyle="--",
        alpha=0.8,
    )

    ax.axhline(0, color="0.80", linewidth=1.0)
    
    ax.set_xlabel("Time inside local window")
    ax.set_ylabel("Z-normalized interest")

    ax.legend(frameon=True, loc="upper right")

    pad = 0.35

    ymin = np.floor((z.min() - pad) * 2) / 2
    ymax = np.ceil((z.max() + pad) * 2) / 2

    ax.set_ylim(ymin, ymax)

    ax.grid(axis="y", color="0.85", linewidth=0.8)
    ax.grid(axis="x", color="0.90", linewidth=0.6, linestyle="--")

    fig.tight_layout()

    fig.savefig(
        path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)


term = choose_sax_example_term(
    panel_norm,
    preferred=SAX_EXAMPLE_TERM,
)

plot_sax_example(
    panel_norm,
    term,
    MARKET_DIR / "fig_sax_example.png",
)

In [54]:
"""
Poster figure generation:
  (1) Cluster shape figure — medoid/centroid curve per cluster, named
      descriptively (shape only, not causal), with unstable clusters
      visually flagged rather than confidently named.
  (2) 4-market x cluster predictive-power heatmap — OOS R^2 as color,
      directional accuracy as annotation, stability flag as a marker,
      so a flashy-but-unstable cell can't be mistaken for a solid finding.

Inputs expected in /mnt/user-data/uploads/:
  cluster_sax_segment_centroid_profiles.csv
  cluster_internal_validation_summary.csv
  POSTER_best_horizon_core_four_metrics.csv

Outputs written to /mnt/user-data/outputs/:
  fig_cluster_shapes_named.png
  fig_predictive_power_heatmap.png
"""
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm

UPLOAD_DIR = Path(r"C:\Python\CSUREMM\output\sax_tests_july_08\graph")
OUT_DIR = MARKET_DIR / "figs"
os.makedirs(OUT_DIR, exist_ok=True)

# A cluster counts as "stable" (safe to name descriptively / feature
# prominently) if its term-level stability score is at or above this.
# Threshold is applied to `stability_median` from the validation summary.
STABILITY_THRESHOLD = 0.40


# ---------------------------------------------------------------------------
# Shared cluster metadata: shape descriptors, stability, descriptive name
# ---------------------------------------------------------------------------

def load_cluster_meta():
    centroids = pd.read_csv(
        f"{UPLOAD_DIR}/cluster_sax_segment_centroid_profiles.csv"
    ).set_index("cluster")
    validation = pd.read_csv(
        f"{UPLOAD_DIR}/cluster_internal_validation_summary.csv"
    ).set_index("cluster")

    seg_cols = [c for c in centroids.columns if c.startswith("sax_seg_mean_")]
    seg_cols = sorted(seg_cols)

    meta = pd.DataFrame(index=centroids.index)
    meta["n_terms"] = validation["n_terms"]
    meta["stability_median"] = validation["stability_median"]
    meta["is_stable"] = meta["stability_median"] >= STABILITY_THRESHOLD

    # --- shape descriptors from the 16-segment centroid profile ---
    profiles = centroids[seg_cols].values.astype(float)
    trend = profiles[:, -4:].mean(axis=1) - profiles[:, :4].mean(axis=1)
    amplitude = profiles.std(axis=1)
    level = profiles.mean(axis=1)
    meta["trend"] = trend
    meta["amplitude"] = amplitude
    meta["level"] = level

    amp_med = np.median(amplitude)
    trend_thresh = 0.02

    base_names = []
    for tr, amp in zip(trend, amplitude):
        cyclical = amp >= amp_med
        if cyclical and tr <= -trend_thresh:
            name = "Seasonal decline"
        elif cyclical and tr >= trend_thresh:
            name = "Seasonal build"
        elif cyclical:
            name = "Seasonal pulse"
        elif tr <= -trend_thresh:
            name = "Slow fade"
        elif tr >= trend_thresh:
            name = "Gradual climb"
        else:
            name = "Flat & steady"
        base_names.append(name)
    meta["base_name"] = base_names

    # disambiguate duplicate names within a group using relative level/amplitude
    final_names = []
    for cl in meta.index:
        group = meta[meta["base_name"] == meta.loc[cl, "base_name"]]
        if len(group) <= 1:
            final_names.append(meta.loc[cl, "base_name"])
            continue
        # rank this cluster's level within the duplicate-name group
        ranked = group["level"].rank(ascending=False, method="first")
        rank = ranked.loc[cl]
        n = len(group)
        if n == 2:
            qualifier = "higher baseline" if rank == 1 else "lower baseline"
        else:
            top_cut = np.ceil(n / 3)
            bottom_cut = n - np.ceil(n / 3)
            if rank <= top_cut:
                qualifier = "higher baseline"
            elif rank > bottom_cut:
                qualifier = "lower baseline"
            else:
                qualifier = "mid baseline"
        final_names.append(f"{meta.loc[cl, 'base_name']} ({qualifier})")
    meta["display_name"] = final_names

    meta["profile"] = list(profiles)
    return meta, seg_cols


# ---------------------------------------------------------------------------
# Figure 1: shape figure with memorable (but honest) names
# ---------------------------------------------------------------------------
def make_shape_figure(meta, seg_cols):

    selected_clusters = [7, 1, 0]

    cluster_names = {
        7: "intermittent surges",
        1: "gradual climb",
        0: "seasonal pulse",
    }

    cluster_terms = {
        7: ["Outlook Email", "Microsoft", "Kahoot"],
        1: ["Elections", "Election Day", "Election Results"],
        0: ["NY Yankees", "Basketball Tonight", "Notre Dame"],
    }

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(16, 5),
        sharey=False,
    )

    x = np.arange(1, len(seg_cols) + 1)

    for ax, cl in zip(axes, selected_clusters):

        row = meta.loc[cl]
        profile = np.asarray(row["profile"], dtype=float)
        spread = 0.35 * float(row["amplitude"]) + 0.01

        # Flexible y-axis for each panel
        ymin = np.min(profile - spread)
        ymax = np.max(profile + spread)

        yrange = ymax - ymin
        pad = max(0.08, 0.15 * yrange)

        ymin -= pad
        ymax += pad

        ymin = np.floor(ymin * 10) / 10
        ymax = np.ceil(ymax * 10) / 10

        yticks = np.linspace(ymin, ymax, 5)

        if ymin < 0 < ymax:
            ax.axhline(
                0,
                color="gray",
                linewidth=1,
                linestyle="--",
                alpha=0.55,
                zorder=0,
            )

        ax.fill_between(
            x,
            profile - spread,
            profile + spread,
            color="steelblue",
            alpha=0.18,
            linewidth=0,
            zorder=1,
        )

        ax.plot(
            x,
            profile,
            color="steelblue",
            linewidth=2.8,
            zorder=2,
        )

        ax.set_title(
            cluster_names[cl],
            fontsize=17,
            fontweight="normal",
            pad=26,
        )

        # ax.text(
        #     0.5,
        #     1.05,
        #     f"Cluster {cl}   n={int(row['n_terms'])}   Stability={row['stability_median']:.2f}",
        #     transform=ax.transAxes,
        #     ha="center",
        #     va="bottom",
        #     fontsize=10,
        #     color="dimgray",
        # )

        # ax.text(
        #     0.5,
        #     -0.12,
        #     "\n".join(cluster_terms[cl]),
        #     transform=ax.transAxes,
        #     ha="center",
        #     va="top",
        #     fontsize=11,
        #     color="black",
        #     linespacing=1.4,
        # )

        ax.set_xlim(1, len(seg_cols))
        ax.set_ylim(ymin, ymax)

        ax.set_xticks([])
        ax.set_yticks(yticks)
        ax.tick_params(axis="y", labelsize=9)

        ax.set_box_aspect(0.60)

        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor("black")
            spine.set_linewidth(1.8)

    plt.subplots_adjust(
        left=0.06,
        right=0.98,
        top=0.82,
        bottom=0.28,
        wspace=0.38,
    )

    fig.savefig(
        f"{OUT_DIR}/fig_cluster_shapes_named_selected.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)

In [55]:
if __name__ == "__main__":
    meta, seg_cols = load_cluster_meta()
    print(meta[["n_terms", "stability_median", "is_stable", "display_name"]])
    make_shape_figure(meta, seg_cols)
    print("Done.")

         n_terms  stability_median  is_stable  \
cluster                                         
0            107          0.641681       True   
1             98          0.644006       True   
2            158          0.194268      False   
3             98          0.549511       True   
4             93          0.348396      False   
5             63          0.523115       True   
6              5          0.274785      False   
7            119          0.842086       True   
8             89          0.546679       True   

                               display_name  
cluster                                      
0           Seasonal pulse (lower baseline)  
1                             Gradual climb  
2        Seasonal decline (higher baseline)  
3            Flat & steady (lower baseline)  
4         Seasonal decline (lower baseline)  
5          Seasonal pulse (higher baseline)  
6             Seasonal pulse (mid baseline)  
7           Flat & steady (higher baseline)  


In [50]:
# ---------------------------------------------------------------------------
# Figure 2: 4-market x cluster predictive-power heatmap
# ---------------------------------------------------------------------------

def make_predictive_heatmap(meta):
    best = pd.read_csv(f"{UPLOAD_DIR}/POSTER_best_horizon_core_four_metrics.csv")
    best["cluster"] = (
        best["cluster_index"].str.extract(r"cluster_(\d+)_index").astype(int)
    )

    markets = ["sp500", "nasdaq100", "dowjones", "russell2000"]
    market_labels = {
        "sp500": "S&P 500", "nasdaq100": "Nasdaq 100",
        "dowjones": "Dow Jones", "russell2000": "Russell 2000",
    }

    cluster_order = meta.sort_values(
        ["is_stable", "stability_median"], ascending=[False, False]
    ).index.tolist()

    r2 = pd.DataFrame(index=cluster_order, columns=markets, dtype=float)
    da = pd.DataFrame(index=cluster_order, columns=markets, dtype=float)
    flag = pd.DataFrame(index=cluster_order, columns=markets, dtype=object)
    horizon = pd.DataFrame(index=cluster_order, columns=markets, dtype=object)

    for _, r in best.iterrows():
        if r["market_index"] not in markets:
            continue
        cl = r["cluster"]
        m = r["market_index"]
        r2.loc[cl, m] = r["poster_oos_r2_vs_lagged_vol_baseline"]
        da.loc[cl, m] = r["poster_directional_accuracy"]
        flag.loc[cl, m] = r["poster_stability_flag"]
        horizon.loc[cl, m] = r["horizon_days"]

    r2_vals = r2.values.astype(float)
    vmax = np.nanmax(np.abs(r2_vals)) if np.isfinite(r2_vals).any() else 1.0
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

    fig, ax = plt.subplots(figsize=(9.5, 0.62 * len(cluster_order) + 1.8))
    im = ax.imshow(r2_vals, cmap="RdBu_r", norm=norm, aspect="auto")

    ax.set_xticks(range(len(markets)))
    ax.set_xticklabels([market_labels[m] for m in markets], fontsize=10)
    row_labels = [
        f"C{cl} — {meta.loc[cl, 'display_name'] if meta.loc[cl, 'is_stable'] else 'provisional'}"
        for cl in cluster_order
    ]
    ax.set_yticks(range(len(cluster_order)))
    ax.set_yticklabels(row_labels, fontsize=9.5)

    flag_style = {
        "stable": dict(marker="o", color="darkgreen", label="stable result"),
        "provisional": dict(marker="^", color="darkorange", label="provisional result"),
        "weak": dict(marker="x", color="dimgray", label="weak / null result"),
    }

    for i, cl in enumerate(cluster_order):
        for j, m in enumerate(markets):
            r2_val = r2.loc[cl, m]
            da_val = da.loc[cl, m]
            f_val = flag.loc[cl, m]
            if pd.isna(r2_val):
                ax.text(j, i, "n/a", ha="center", va="center", fontsize=8, color="gray")
                continue
            ax.text(
                j, i - 0.16, f"R\u00b2={r2_val:.2f}",
                ha="center", va="center", fontsize=8.5,
                color="white" if abs(r2_val) > vmax * 0.55 else "black",
                fontweight="bold",
            )
            ax.text(
                j, i + 0.16, f"DA={da_val:.0%}",
                ha="center", va="center", fontsize=7.5,
                color="white" if abs(r2_val) > vmax * 0.55 else "black",
            )
            style = flag_style.get(f_val, flag_style["weak"])
            ax.scatter(
                j + 0.36, i - 0.36, marker=style["marker"], color=style["color"],
                s=55, edgecolor="white", linewidth=0.6, zorder=5, clip_on=False,
            )
        # left-margin stability marker for cluster composition
        row_stable = meta.loc[cl, "is_stable"]
        ax.text(
            -0.62, i, "\u25CF" if row_stable else "\u25B3",
            ha="center", va="center", fontsize=11,
            color="darkgreen" if row_stable else "darkorange",
        )

    ax.set_xticks(np.arange(-0.5, len(markets), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(cluster_order), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.5)
    ax.tick_params(which="minor", bottom=False, left=False)

    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.03)
    cbar.set_label("OOS R\u00b2 vs. lagged-volatility baseline\n(best horizon per cluster/market)", fontsize=8.5)

    legend_handles = [
        mpatches.Patch(color="none", label="Row marker:"),
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="darkgreen", markersize=8, label="stable cluster (composition)"),
        plt.Line2D([0], [0], marker="^", color="w", markerfacecolor="darkorange", markersize=8, label="provisional cluster (composition)"),
        mpatches.Patch(color="none", label="Cell marker (result):"),
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="darkgreen", markersize=7, label="stable result"),
        plt.Line2D([0], [0], marker="^", color="w", markerfacecolor="darkorange", markersize=7, label="provisional result"),
        plt.Line2D([0], [0], marker="x", color="dimgray", markersize=7, label="weak / null result"),
    ]
    ax.legend(
        handles=legend_handles, loc="upper left", bbox_to_anchor=(1.28, 1.0),
        fontsize=8, frameon=False, handletextpad=0.6, labelspacing=0.6,
    )

    ax.set_title(
        "Out-of-sample predictive power by cluster and market index\n"
        "(best horizon shown per cell; R\u00b2 and directional accuracy reported for every cell, weak/null included)",
        fontsize=11.5, fontweight="bold", pad=14,
    )

    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/fig_predictive_power_heatmap.png", dpi=220, bbox_inches="tight")
    plt.close(fig)


if __name__ == "__main__":
    meta, seg_cols = load_cluster_meta()
    print(meta[["n_terms", "stability_median", "is_stable", "display_name"]])
    make_shape_figure(meta, seg_cols)
    make_predictive_heatmap(meta)
    print("Done.")

         n_terms  stability_median  is_stable  \
cluster                                         
0            107          0.641681       True   
1             98          0.644006       True   
2            158          0.194268      False   
3             98          0.549511       True   
4             93          0.348396      False   
5             63          0.523115       True   
6              5          0.274785      False   
7            119          0.842086       True   
8             89          0.546679       True   

                               display_name  
cluster                                      
0           Seasonal pulse (lower baseline)  
1                             Gradual climb  
2        Seasonal decline (higher baseline)  
3            Flat & steady (lower baseline)  
4         Seasonal decline (lower baseline)  
5          Seasonal pulse (higher baseline)  
6             Seasonal pulse (mid baseline)  
7           Flat & steady (higher baseline)  
